<a href="https://colab.research.google.com/github/belybutcher/CU-AI-Nexus-2026/blob/main/BUSI_Breast_Ultrasound_Classification_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AI-Powered Ultrasound Assistant — Disease Classification Module

## Breast Ultrasound Image Classification (BUSI Dataset)

**Hackathon:** Cairo University AI Hackathon  
**Team:** [Your Team Name]  
**Challenge:** Classify breast ultrasound images into three categories:
- **Normal** — Healthy breast tissue
- **Benign** — Non-cancerous abnormalities
- **Malignant** — Cancerous tumors

**Notebook Structure:**
1. Environment Setup
2. Google Drive Setup
3. Dataset Download & Inspection
4. Exploratory Data Analysis (EDA)
5. Data Preprocessing
6. Model Selection
7. Build Classification Model
8. Training Pipeline
9. Model Evaluation
10. Error Analysis
11. Prediction Function
12. Export & Inference

---

> **Note:** Run each cell sequentially. Markdown explanations are provided before every code block.



---

# Section 1: Environment Setup

This section prepares the computational environment for training a deep learning model on breast ultrasound images.

### What we do here:
1. **Install required packages** — PyTorch, torchvision, and utilities for data analysis and visualization.
2. **Import libraries** — Organized into standard library, data science, image processing, deep learning, and visualization.
3. **Check GPU availability** — Ensure a CUDA-enabled GPU is available for accelerated training.
4. **Set random seeds** — Ensure reproducibility across runs.
5. **Configure project paths** — Centralized path management for easy configuration.

### Assumptions:
- You are running this in **Google Colab** with a **GPU runtime** (T4/V100/A100).
- The BUSI dataset will be stored in your Google Drive under `MyDrive/Datasets/BUSI/`.


In [ ]:

# ============================================================
# 1.1 Install Required Packages
# ============================================================
# Google Colab comes with PyTorch pre-installed, but we ensure
# all dependencies are up-to-date.

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q scikit-learn matplotlib seaborn pandas numpy pillow tqdm opencv-python

print("All packages installed successfully!")


All packages installed successfully!


In [ ]:

# ============================================================
# 1.2 Import Libraries
# ============================================================
# Imports are organized by category for clarity and maintainability.

# --- Standard Library ---
import os
import sys
import random
import warnings
import itertools
import json
import time
import shutil
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Union, Optional, Callable

# --- Data Science ---
import numpy as np
import pandas as pd

# --- Image Processing ---
from PIL import Image, ImageOps, ImageFilter, ImageStat
import cv2

# --- Deep Learning (PyTorch) ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torch.cuda.amp import autocast, GradScaler
import torchvision
from torchvision import transforms, models
from torchvision.models import efficientnet_b0, resnet18, densenet121, mobilenet_v3_small

# --- Machine Learning Utilities ---
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# --- Progress Bar ---
from tqdm.notebook import tqdm

# --- Suppress warnings for cleaner output ---
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print(f"Python version: {sys.version}")


All libraries imported successfully!
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:

# ============================================================
# 1.3 Check GPU Availability
# ============================================================
# Training deep learning models on images requires a GPU.
# This cell verifies that a CUDA-enabled GPU is available.

print("=" * 60)
print("GPU INFORMATION")
print("=" * 60)

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)  # GB
    print(f"Device: GPU (CUDA available)")
    print(f"GPU Name: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("Please enable GPU runtime: Runtime > Change runtime type > GPU")

print(f"Using device: {device}")
print("=" * 60)


GPU INFORMATION
Device: GPU (CUDA available)
GPU Name: Tesla T4
GPU Memory: 14.56 GB
CUDA Version: 12.8
PyTorch Version: 2.11.0+cu128
Using device: cuda


In [ ]:

# ============================================================
# 1.4 Set Random Seeds for Reproducibility
# ============================================================
# Setting seeds ensures that the results are reproducible
# across different runs, which is critical for experiments.

def set_seed(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across all libraries.

    Args:
        seed: Integer seed value. Default is 42 (common in ML).
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU setups
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Random seed set to {seed} for reproducibility.")

# Set the seed
SEED = 42
set_seed(SEED)


Random seed set to 42 for reproducibility.


In [ ]:

# ============================================================
# 1.5 Configure Project Paths
# ============================================================
# All paths are centralized here for easy configuration.
# In Google Colab, we use /content/ for temporary storage
# and Google Drive for persistent storage.

# --- Base Paths ---
BASE_DIR = Path("/content")  # Colab working directory
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_DATASET_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "Datasets" / "BUSI"
LOCAL_DATASET_DIR = BASE_DIR / "BUSI"

# --- Subdirectory Paths ---
DATASET_IMAGE_DIR = LOCAL_DATASET_DIR / "Dataset"
CHECKPOINT_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "BUSI_Checkpoints"
RESULTS_DIR = BASE_DIR / "results"

# --- Model Checkpoint Paths ---
BEST_MODEL_PATH = CHECKPOINT_DIR / "classification_model.pth"
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_checkpoint.pt"

# --- Create local directories ---
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project paths configured:")
print(f"  Local dataset dir: {LOCAL_DATASET_DIR}")
print(f"  Drive dataset dir: {DRIVE_DATASET_DIR}")
print(f"  Checkpoint dir:    {CHECKPOINT_DIR}")
print(f"  Results dir:       {RESULTS_DIR}")

# --- Class Configuration ---
CLASS_NAMES = ["normal", "benign", "malignant"]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}

print(f"Classes: {CLASS_NAMES}")
print(f"Class to index mapping: {CLASS_TO_IDX}")


Project paths configured:
  Local dataset dir: /content/BUSI
  Drive dataset dir: /content/drive/MyDrive/Datasets/BUSI
  Checkpoint dir:    /content/drive/MyDrive/BUSI_Checkpoints
  Results dir:       /content/results
Classes: ['normal', 'benign', 'malignant']
Class to index mapping: {'normal': 0, 'benign': 1, 'malignant': 2}



---

# Section 2: Google Drive Setup

Google Drive is used for **persistent storage** of the dataset and model checkpoints. Colab's local storage (`/content/`) is temporary and resets when the runtime disconnects.

### What we do here:
1. **Mount Google Drive** — Link your Drive to the Colab filesystem.
2. **Explain the recommended dataset directory structure**.
3. **Create checkpoint directory** in Drive for saving models.

### Recommended Dataset Directory Structure:
```
MyDrive/
└── Datasets/
    └── BUSI/
        └── Dataset/
            ├── normal/
            │   ├── normal-001.png
            │   ├── normal-001_mask.png
            │   ├── normal-002.png
            │   └── normal-002_mask.png
            ├── benign/
            │   ├── benign-001.png
            │   ├── benign-001_mask.png
            │   └── ...
            └── malignant/
                ├── malignant-001.png
                ├── malignant-001_mask.png
                └── ...
```

**Note:** Each image may have a corresponding `_mask.png` file containing the lesion segmentation mask.


In [ ]:

# ============================================================
# 2.1 Mount Google Drive
# ============================================================
# This will prompt you to authorize access to your Google Drive.
# Click the link, sign in, and paste the authorization code.

from google.colab import drive

drive.mount('/content/drive')
print("Google Drive mounted successfully!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!


In [ ]:

# ============================================================
# 2.2 Create Checkpoint Directory in Drive
# ============================================================
# Model checkpoints will be saved here for persistence across sessions.

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint directory ready: {CHECKPOINT_DIR}")
print(f"Directory exists: {CHECKPOINT_DIR.exists()}")

# List contents of the dataset directory if it exists
if DRIVE_DATASET_DIR.exists():
    print(f"Dataset directory found: {DRIVE_DATASET_DIR}")
    print("Contents:")
    for item in sorted(DRIVE_DATASET_DIR.iterdir()):
        print(f"  {'[DIR] ' if item.is_dir() else '[FILE]'} {item.name}")
else:
    print(f"Dataset directory NOT found: {DRIVE_DATASET_DIR}")
    print("You will need to upload the BUSI dataset to this location.")
    print("Alternatively, we can download it automatically in the next section.")


Checkpoint directory ready: /content/drive/MyDrive/BUSI_Checkpoints
Directory exists: True
Dataset directory NOT found: /content/drive/MyDrive/Datasets/BUSI
You will need to upload the BUSI dataset to this location.
Alternatively, we can download it automatically in the next section.



---

# Section 3: Dataset Download and Inspection

The **BUSI (Breast Ultrasound Images)** dataset contains ultrasound images classified into three categories: Normal, Benign, and Malignant. Some images include expert-annotated segmentation masks.

### Dataset Download Options:
1. **From Google Drive** (recommended if you already have it)
2. **From Kaggle** — Download directly using the Kaggle API
3. **Manual upload** — Upload via Colab's file manager

### What this section covers:
- Automatic dataset download from Kaggle (if not present in Drive)
- Verification of folder structure
- Verification of class labels
- Counting images and masks
- Matching images with masks
- Detecting missing masks, duplicates, corrupted files
- Generating a comprehensive verification report

### Dataset Source:
- Original Paper: *"Dataset of breast ultrasound images"* by Al-Dhabyani et al.
- Images: 780 images across three classes
- Masks: Segmentation masks for benign and malignant cases


In [ ]:

# ============================================================
# 3.1 Dataset Download (from Kaggle if needed)
# ============================================================
# If the dataset is not in your Google Drive, we download it from Kaggle.
# You need a Kaggle API key (kaggle.json) for this.

import zipfile

def download_busi_from_kaggle():
    """
    Download the BUSI dataset from Kaggle.
    Requires kaggle.json API key to be configured.
    """
    print("Attempting to download BUSI dataset from Kaggle...")

    # Check if kaggle is installed
    try:
        import kaggle
    except ImportError:
        print("Installing Kaggle API...")
        !pip install -q kaggle

    # Upload kaggle.json if not present
    kaggle_dir = Path("/root/.kaggle")
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    if not (kaggle_dir / "kaggle.json").exists():
        print("Please upload your kaggle.json API key file:")
        from google.colab import files
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            shutil.move('kaggle.json', kaggle_dir / 'kaggle.json')
            (kaggle_dir / 'kaggle.json').chmod(0o600)
            print("Kaggle API key configured.")
        else:
            print("ERROR: kaggle.json not uploaded. Cannot download from Kaggle.")
            return False

    # Download dataset
    !kaggle datasets download -d aryashah2k/breast-ultrasound-images-dataset -p {LOCAL_DATASET_DIR} --unzip
    print("Dataset downloaded from Kaggle successfully!")
    return True

def extract_dataset():
    """Extract and organize the downloaded dataset."""
    # Find and extract zip files if any
    for zip_file in LOCAL_DATASET_DIR.glob("*.zip"):
        print(f"Extracting {zip_file.name}...")
        with zipfile.ZipFile(zip_file, 'r') as z:
            z.extractall(LOCAL_DATASET_DIR)
        zip_file.unlink()  # Remove zip after extraction
        print(f"Extracted and removed {zip_file.name}")

# Check if dataset exists in Drive, otherwise download
if DRIVE_DATASET_DIR.exists() and any(DRIVE_DATASET_DIR.iterdir()):
    print("Dataset found in Google Drive. Copying to local storage...")
    shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR, dirs_exist_ok=True)
    print(f"Dataset copied to: {LOCAL_DATASET_DIR}")
elif LOCAL_DATASET_DIR.exists() and any(LOCAL_DATASET_DIR.iterdir()):
    print(f"Dataset already in local storage: {LOCAL_DATASET_DIR}")
else:
    print("Dataset not found. Attempting download from Kaggle...")
    LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    success = download_busi_from_kaggle()
    if success:
        extract_dataset()
        # Copy to Drive for future use
        if DRIVE_MOUNT_POINT.exists():
            DRIVE_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(LOCAL_DATASET_DIR, DRIVE_DATASET_DIR, dirs_exist_ok=True)
            print(f"Dataset copied to Google Drive for future use.")
    else:
        print("Download failed. Please upload the dataset manually.")
        print("Upload the BUSI folder to: /content/BUSI/")


Dataset not found. Attempting download from Kaggle...
Attempting to download BUSI dataset from Kaggle...
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


NameError: name 'exit' is not defined

In [ ]:

# ============================================================
# 3.2 Dataset Inspection and Verification
# ============================================================
# This cell performs comprehensive verification of the dataset:
# - Folder structure validation
# - File counting and matching
# - Corruption detection
# - Duplicate detection

class DatasetInspector:
    """
    Comprehensive dataset inspector for the BUSI dataset.
    Verifies structure, counts files, detects issues.
    """

    def __init__(self, dataset_dir: Path):
        self.dataset_dir = Path(dataset_dir)
        self.classes = ["normal", "benign", "malignant"]
        self.image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
        self.report = {}
        self.issues = []

    def verify_structure(self) -> Dict:
        """Verify that the expected folder structure exists."""
        print("=" * 60)
        print("DATASET STRUCTURE VERIFICATION")
        print("=" * 60)

        structure_ok = True
        class_dirs = {}

        for cls in self.classes:
            class_dir = self.dataset_dir / cls
            if class_dir.exists():
                print(f"[OK] Class directory found: {cls}/")
                class_dirs[cls] = class_dir
            else:
                # Try alternate locations
                alt_dirs = list(self.dataset_dir.rglob(cls))
                if alt_dirs:
                    class_dir = alt_dirs[0]
                    print(f"[OK] Class directory found (alternate path): {class_dir}")
                    class_dirs[cls] = class_dir
                else:
                    print(f"[MISSING] Class directory NOT found: {cls}/")
                    structure_ok = False

        self.class_dirs = class_dirs
        self.report['structure_ok'] = structure_ok
        self.report['class_dirs'] = {k: str(v) for k, v in class_dirs.items()}

        print(f"Overall structure: {'VALID' if structure_ok else 'INVALID'}")
        return self.report

    def count_files(self) -> Dict:
        """Count images and masks per class."""
        print("\n" + "=" * 60)
        print("FILE COUNTING")
        print("=" * 60)

        stats = {}
        total_images = 0
        total_masks = 0

        for cls, class_dir in self.class_dirs.items():
            files = list(class_dir.iterdir()) if class_dir.exists() else []

            # Separate images and masks
            images = [f for f in files if f.suffix.lower() in self.image_extensions
                      and '_mask' not in f.stem]
            masks = [f for f in files if f.suffix.lower() in self.image_extensions
                     and '_mask' in f.stem]

            stats[cls] = {
                'images': len(images),
                'masks': len(masks),
                'image_names': sorted([f.name for f in images]),
                'mask_names': sorted([f.name for f in masks])
            }
            total_images += len(images)
            total_masks += len(masks)

            print(f"{cls:12s}: {len(images):4d} images, {len(masks):4d} masks")

        print(f"{'TOTAL':12s}: {total_images:4d} images, {total_masks:4d} masks")

        self.report['file_counts'] = stats
        self.report['total_images'] = total_images
        self.report['total_masks'] = total_masks

        return stats

    def match_images_masks(self) -> Dict:
        """Match each image with its corresponding mask and detect issues."""
        print("\n" + "=" * 60)
        print("IMAGE-MASK MATCHING")
        print("=" * 60)

        matching_results = {}

        for cls, class_dir in self.class_dirs.items():
            if not class_dir.exists():
                continue

            files = list(class_dir.iterdir())
            images = {f.stem: f for f in files
                      if f.suffix.lower() in self.image_extensions and '_mask' not in f.stem}
            masks = {f.stem.replace('_mask', ''): f for f in files
                     if f.suffix.lower() in self.image_extensions and '_mask' in f.stem}

            matched = []
            missing_masks = []
            extra_masks = []

            for img_stem, img_path in images.items():
                if img_stem in masks:
                    matched.append({
                        'image': img_path.name,
                        'mask': masks[img_stem].name
                    })
                else:
                    missing_masks.append(img_path.name)

            for mask_stem, mask_path in masks.items():
                if mask_stem not in images:
                    extra_masks.append(mask_path.name)

            matching_results[cls] = {
                'matched': len(matched),
                'missing_masks': missing_masks,
                'extra_masks': extra_masks
            }

            print(f"\n{cls.upper()}:")
            print(f"  Matched image-mask pairs: {len(matched)}")
            if missing_masks:
                print(f"  Images WITHOUT masks: {len(missing_masks)}")
                for m in missing_masks[:5]:
                    print(f"    - {m}")
                if len(missing_masks) > 5:
                    print(f"    ... and {len(missing_masks)-5} more")
                self.issues.extend([f"{cls}: Missing mask for {m}" for m in missing_masks])
            if extra_masks:
                print(f"  Masks WITHOUT images: {len(extra_masks)}")
                for m in extra_masks[:5]:
                    print(f"    - {m}")
                self.issues.extend([f"{cls}: Extra mask {m}" for m in extra_masks])

        self.report['mask_matching'] = matching_results
        return matching_results

    def detect_duplicates(self) -> List:
        """Detect duplicate files based on file hash."""
        print("\n" + "=" * 60)
        print("DUPLICATE DETECTION")
        print("=" * 60)

        import hashlib

        file_hashes = defaultdict(list)
        duplicates = []

        for cls, class_dir in self.class_dirs.items():
            if not class_dir.exists():
                continue
            for f in class_dir.iterdir():
                if f.suffix.lower() in self.image_extensions:
                    file_hash = hashlib.md5(f.read_bytes()).hexdigest()
                    file_hashes[file_hash].append(str(f))

        for file_hash, paths in file_hashes.items():
            if len(paths) > 1:
                duplicates.append({'hash': file_hash, 'files': paths})
                print(f"[DUPLICATE] {len(paths)} files with same content:")
                for p in paths:
                    print(f"    - {p}")
                self.issues.append(f"Duplicate files detected: {paths}")

        if not duplicates:
            print("No duplicate files found.")

        self.report['duplicates'] = duplicates
        return duplicates

    def detect_corrupted(self) -> List:
        """Detect corrupted or unreadable images."""
        print("\n" + "=" * 60)
        print("CORRUPTION DETECTION")
        print("=" * 60)

        corrupted = []

        for cls, class_dir in self.class_dirs.items():
            if not class_dir.exists():
                continue
            files = [f for f in class_dir.iterdir()
                     if f.suffix.lower() in self.image_extensions and '_mask' not in f.stem]

            for f in tqdm(files, desc=f"Checking {cls}", leave=False):
                try:
                    img = Image.open(f)
                    img.verify()  # Verify without loading
                    # Also try loading
                    img = Image.open(f)
                    img.load()
                except Exception as e:
                    corrupted.append({'file': str(f), 'error': str(e)})
                    print(f"[CORRUPTED] {f.name}: {e}")
                    self.issues.append(f"Corrupted file: {f.name} - {e}")

        if not corrupted:
            print("No corrupted images found.")

        self.report['corrupted_files'] = corrupted
        return corrupted

    def generate_summary_table(self) -> pd.DataFrame:
        """Generate a summary table of the dataset."""
        print("\n" + "=" * 60)
        print("DATASET SUMMARY TABLE")
        print("=" * 60)

        data = []
        total_imgs = 0
        total_msks = 0

        for cls in self.classes:
            if cls in self.report.get('file_counts', {}):
                imgs = self.report['file_counts'][cls]['images']
                msks = self.report['file_counts'][cls]['masks']
                total_imgs += imgs
                total_msks += msks
                pct = (imgs / total_imgs * 100) if total_imgs > 0 else 0
                data.append({
                    'Class': cls.capitalize(),
                    'Images': imgs,
                    'Masks': msks,
                    'Percentage': f"{pct:.1f}%"
                })

        df = pd.DataFrame(data)
        print(df.to_string(index=False))
        print(f"\nTotal Images: {total_imgs}")
        print(f"Total Masks:  {total_msks}")

        self.report['summary_table'] = df.to_dict('records')
        return df

    def generate_verification_report(self) -> str:
        """Generate a comprehensive verification report."""
        print("\n" + "=" * 60)
        print("VERIFICATION REPORT")
        print("=" * 60)

        report_lines = [
            "BUSI DATASET VERIFICATION REPORT",
            "=" * 40,
            f"Dataset Path: {self.dataset_dir}",
            f"Structure Valid: {self.report.get('structure_ok', False)}",
            f"Total Images: {self.report.get('total_images', 0)}",
            f"Total Masks: {self.report.get('total_masks', 0)}",
            "",
            "Class Distribution:",
        ]

        for cls in self.classes:
            if cls in self.report.get('file_counts', {}):
                imgs = self.report['file_counts'][cls]['images']
                report_lines.append(f"  {cls}: {imgs} images")

        report_lines.extend([
            "",
            f"Duplicate Files: {len(self.report.get('duplicates', []))}",
            f"Corrupted Files: {len(self.report.get('corrupted_files', []))}",
            f"Total Issues Found: {len(self.issues)}",
        ])

        if self.issues:
            report_lines.extend(["", "Issues:", "-" * 20])
            for issue in self.issues[:20]:
                report_lines.append(f"  - {issue}")
            if len(self.issues) > 20:
                report_lines.append(f"  ... and {len(self.issues)-20} more issues")

        report_text = "\n".join(report_lines)
        print(report_text)

        # Save report to file
        report_path = RESULTS_DIR / "dataset_verification_report.txt"
        with open(report_path, 'w') as f:
            f.write(report_text)
        print(f"\nReport saved to: {report_path}")

        return report_text

    def run_full_inspection(self):
        """Run all inspection steps."""
        print("\n" + "=" * 60)
        print("STARTING FULL DATASET INSPECTION")
        print("=" * 60)

        self.verify_structure()
        self.count_files()
        self.match_images_masks()
        self.detect_duplicates()
        self.detect_corrupted()
        self.generate_summary_table()
        self.generate_verification_report()

        print("\n" + "=" * 60)
        print("INSPECTION COMPLETE")
        print("=" * 60)
        return self.report


# Run the inspection
inspector = DatasetInspector(LOCAL_DATASET_DIR / "Dataset" if (LOCAL_DATASET_DIR / "Dataset").exists() else LOCAL_DATASET_DIR)
inspection_report = inspector.run_full_inspection()



---

# Section 4: Exploratory Data Analysis (EDA)

Comprehensive analysis of the BUSI dataset to understand:
- Class distribution and balance
- Image properties (size, resolution, format)
- Pixel-level statistics
- Image quality metrics
- Visual characteristics of each class
- Segmentation mask properties

EDA informs preprocessing decisions, augmentation strategies, and model architecture choices.



## 4.1 Class Distribution

Understanding class balance is critical because:
- **Imbalanced datasets** can cause models to bias toward majority classes
- **Class weights** or **sampling strategies** may be needed
- **Evaluation metrics** must account for imbalance (F1, not just accuracy)

We calculate the number of images per class, percentages, and visualize the distribution.


In [ ]:

# ============================================================
# 4.1 Class Distribution Analysis
# ============================================================

def analyze_class_distribution(dataset_dir: Path) -> pd.DataFrame:
    """
    Analyze and visualize the class distribution of the dataset.

    Args:
        dataset_dir: Path to the dataset root directory.

    Returns:
        DataFrame with class distribution statistics.
    """
    classes = ["normal", "benign", "malignant"]
    class_counts = {}

    # Find class directories (handle nested structure)
    class_dirs = {}
    for cls in classes:
        # Try direct path first
        direct_path = dataset_dir / cls
        if direct_path.exists():
            class_dirs[cls] = direct_path
        else:
            # Search recursively
            matches = list(dataset_dir.rglob(cls))
            if matches:
                class_dirs[cls] = matches[0]

    # Count images per class (exclude masks)
    for cls, class_dir in class_dirs.items():
        if class_dir.exists():
            files = [f for f in class_dir.iterdir()
                     if f.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}
                     and '_mask' not in f.stem]
            class_counts[cls] = len(files)
        else:
            class_counts[cls] = 0

    total = sum(class_counts.values())
    data = []
    for cls in classes:
        count = class_counts.get(cls, 0)
        pct = (count / total * 100) if total > 0 else 0
        data.append({
            'Class': cls.capitalize(),
            'Count': count,
            'Percentage': pct
        })

    df = pd.DataFrame(data)

    # --- Visualization ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    bars = axes[0].bar(df['Class'], df['Count'], color=colors, edgecolor='black', linewidth=1.2)
    axes[0].set_title('Class Distribution (Bar Chart)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Class', fontsize=12)
    axes[0].set_ylabel('Number of Images', fontsize=12)
    axes[0].grid(axis='y', alpha=0.3)

    # Add value labels on bars
    for bar, count, pct in zip(bars, df['Count'], df['Percentage']):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                     f'{count}\n({pct:.1f}%)',
                     ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Pie chart
    wedges, texts, autotexts = axes[1].pie(
        df['Count'], labels=df['Class'], autopct='%1.1f%%',
        colors=colors, startangle=90, explode=[0.02, 0.02, 0.02],
        shadow=True, textprops={'fontsize': 11}
    )
    axes[1].set_title('Class Distribution (Pie Chart)', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Balance assessment
    print("\n" + "=" * 60)
    print("CLASS BALANCE ASSESSMENT")
    print("=" * 60)
    print(df.to_string(index=False))

    max_count = df['Count'].max()
    min_count = df['Count'].min()
    imbalance_ratio = max_count / min_count if min_count > 0 else float('inf')

    print(f"\nTotal images: {total}")
    print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")

    if imbalance_ratio < 1.5:
        print("Dataset is relatively BALANCED.")
    elif imbalance_ratio < 3:
        print("Dataset is MODERATELY IMBALANCED. Consider class weighting.")
    else:
        print("Dataset is HIGHLY IMBALANCED. Strong mitigation needed.")

    return df

# Run class distribution analysis
dataset_root = LOCAL_DATASET_DIR / "Dataset" if (LOCAL_DATASET_DIR / "Dataset").exists() else LOCAL_DATASET_DIR
class_dist_df = analyze_class_distribution(dataset_root)



## 4.2 Image Properties

Analyzing image dimensions and formats is essential because:
- **Input size** determines model architecture and computational cost
- **Aspect ratio** informs resizing strategy (stretch vs. crop vs. pad)
- **Color channels** determine preprocessing (grayscale → 3-channel conversion)
- **Resolution variation** indicates whether resizing is needed

We analyze width, height, aspect ratio, format, and color channels across all images.


In [ ]:

# ============================================================
# 4.2 Image Properties Analysis
# ============================================================

def analyze_image_properties(dataset_dir: Path) -> pd.DataFrame:
    """
    Analyze image properties: dimensions, format, color channels, aspect ratio.

    Args:
        dataset_dir: Path to the dataset root directory.

    Returns:
        DataFrame with image property statistics per class.
    """
    classes = ["normal", "benign", "malignant"]
    properties = []

    print("Collecting image properties...")

    for cls in classes:
        # Find class directory
        class_dir = dataset_dir / cls
        if not class_dir.exists():
            matches = list(dataset_dir.rglob(cls))
            class_dir = matches[0] if matches else None

        if class_dir is None or not class_dir.exists():
            continue

        image_files = [f for f in class_dir.iterdir()
                       if f.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}
                       and '_mask' not in f.stem]

        for img_path in tqdm(image_files, desc=f"Analyzing {cls}", leave=False):
            try:
                with Image.open(img_path) as img:
                    width, height = img.size
                    mode = img.mode  # 'L', 'RGB', 'RGBA', etc.
                    aspect_ratio = width / height
                    file_size_kb = img_path.stat().st_size / 1024

                    properties.append({
                        'class': cls,
                        'filename': img_path.name,
                        'width': width,
                        'height': height,
                        'channels': mode,
                        'aspect_ratio': aspect_ratio,
                        'file_size_kb': file_size_kb
                    })
            except Exception as e:
                print(f"Error reading {img_path}: {e}")

    df = pd.DataFrame(properties)

    # --- Summary Statistics ---
    print("\n" + "=" * 60)
    print("IMAGE PROPERTIES SUMMARY")
    print("=" * 60)

    summary = df.groupby('class').agg({
        'width': ['min', 'max', 'mean', 'std'],
        'height': ['min', 'max', 'mean', 'std'],
        'aspect_ratio': ['min', 'max', 'mean', 'std'],
        'file_size_kb': ['mean']
    }).round(2)
    print(summary)

    # --- Overall Statistics ---
    print("\n--- OVERALL STATISTICS ---")
    print(f"Unique widths:  {sorted(df['width'].unique())}")
    print(f"Unique heights: {sorted(df['height'].unique())}")
    print(f"Average width:  {df['width'].mean():.1f}")
    print(f"Average height: {df['height'].mean():.1f}")
    print(f"Min resolution: {df['width'].min()} x {df['height'].min()}")
    print(f"Max resolution: {df['width'].max()} x {df['height'].max()}")

    # Channel distribution
    print(f"\nColor channels:")
    print(df['channels'].value_counts())

    # Unique size combinations
    size_combos = df.groupby(['width', 'height']).size().reset_index(name='count')
    size_combos = size_combos.sort_values('count', ascending=False)
    print(f"\nTop 10 most common resolutions:")
    print(size_combos.head(10).to_string(index=False))

    # --- Visualizations ---
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    # Width distribution
    for cls, color in zip(classes, ['#2ecc71', '#f39c12', '#e74c3c']):
        cls_data = df[df['class'] == cls]['width']
        axes[0, 0].hist(cls_data, bins=20, alpha=0.6, label=cls.capitalize(), color=color, edgecolor='black')
    axes[0, 0].set_title('Width Distribution', fontweight='bold')
    axes[0, 0].set_xlabel('Width (pixels)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    # Height distribution
    for cls, color in zip(classes, ['#2ecc71', '#f39c12', '#e74c3c']):
        cls_data = df[df['class'] == cls]['height']
        axes[0, 1].hist(cls_data, bins=20, alpha=0.6, label=cls.capitalize(), color=color, edgecolor='black')
    axes[0, 1].set_title('Height Distribution', fontweight='bold')
    axes[0, 1].set_xlabel('Height (pixels)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    # Aspect ratio distribution
    for cls, color in zip(classes, ['#2ecc71', '#f39c12', '#e74c3c']):
        cls_data = df[df['class'] == cls]['aspect_ratio']
        axes[0, 2].hist(cls_data, bins=20, alpha=0.6, label=cls.capitalize(), color=color, edgecolor='black')
    axes[0, 2].set_title('Aspect Ratio Distribution', fontweight='bold')
    axes[0, 2].set_xlabel('Aspect Ratio (W/H)')
    axes[0, 2].set_ylabel('Frequency')
    axes[0, 2].legend()
    axes[0, 2].grid(alpha=0.3)

    # Scatter: Width vs Height
    for cls, color in zip(classes, ['#2ecc71', '#f39c12', '#e74c3c']):
        cls_data = df[df['class'] == cls]
        axes[1, 0].scatter(cls_data['width'], cls_data['height'],
                          alpha=0.5, label=cls.capitalize(), color=color, s=30)
    axes[1, 0].set_title('Width vs Height', fontweight='bold')
    axes[1, 0].set_xlabel('Width (pixels)')
    axes[1, 0].set_ylabel('Height (pixels)')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    # Resolution (pixel count) distribution
    df['resolution'] = df['width'] * df['height']
    for cls, color in zip(classes, ['#2ecc71', '#f39c12', '#e74c3c']):
        cls_data = df[df['class'] == cls]['resolution']
        axes[1, 1].hist(cls_data, bins=20, alpha=0.6, label=cls.capitalize(), color=color, edgecolor='black')
    axes[1, 1].set_title('Resolution Distribution', fontweight='bold')
    axes[1, 1].set_xlabel('Resolution (pixels)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

    # File size distribution
    for cls, color in zip(classes, ['#2ecc71', '#f39c12', '#e74c3c']):
        cls_data = df[df['class'] == cls]['file_size_kb']
        axes[1, 2].hist(cls_data, bins=20, alpha=0.6, label=cls.capitalize(), color=color, edgecolor='black')
    axes[1, 2].set_title('File Size Distribution', fontweight='bold')
    axes[1, 2].set_xlabel('File Size (KB)')
    axes[1, 2].set_ylabel('Frequency')
    axes[1, 2].legend()
    axes[1, 2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'image_properties.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Recommendation
    print("\n" + "=" * 60)
    print("RECOMMENDATION")
    print("=" * 60)
    print("Based on the analysis, the recommended input size is: 224 x 224")
    print("This is the standard input size for EfficientNet and most transfer learning models.")
    print("All images will be resized to 224x224 during preprocessing.")

    return df

# Run image properties analysis
img_props_df = analyze_image_properties(dataset_root)



## 4.3 Pixel Statistics

Understanding pixel-level statistics is crucial for **normalization**:
- **Mean and standard deviation** are used to normalize images during preprocessing
- **Intensity distribution** reveals whether images are consistently bright/dark
- **Histogram shape** indicates contrast characteristics

Normalization ensures that the model receives consistent input distributions, which improves training stability and convergence speed. For medical images, using **dataset-specific statistics** (rather than ImageNet defaults) is recommended because ultrasound images have very different intensity distributions from natural photographs.


In [ ]:

# ============================================================
# 4.3 Pixel Statistics
# ============================================================

def analyze_pixel_statistics(dataset_dir: Path, sample_limit: int = 500) -> Dict:
    """
    Compute pixel-level statistics across the dataset.
    Samples images randomly for efficiency if dataset is large.

    Args:
        dataset_dir: Path to dataset root.
        sample_limit: Max number of images to sample per class.

    Returns:
        Dictionary of pixel statistics per class and overall.
    """
    classes = ["normal", "benign", "malignant"]
    all_pixels = []
    class_pixels = {cls: [] for cls in classes}

    print("Collecting pixel statistics (this may take a moment)...")

    for cls in classes:
        class_dir = dataset_dir / cls
        if not class_dir.exists():
            matches = list(dataset_dir.rglob(cls))
            class_dir = matches[0] if matches else None
        if class_dir is None:
            continue

        image_files = [f for f in class_dir.iterdir()
                       if f.suffix.lower() in {'.png', '.jpg', '.jpeg'}
                       and '_mask' not in f.stem]

        # Sample if too many
        if len(image_files) > sample_limit:
            random.seed(42)
            image_files = random.sample(image_files, sample_limit)

        for img_path in tqdm(image_files, desc=f"Processing {cls}", leave=False):
            try:
                img = Image.open(img_path).convert('L')  # Grayscale
                arr = np.array(img, dtype=np.float32)
                class_pixels[cls].extend(arr.flatten())
                all_pixels.extend(arr.flatten())
            except Exception as e:
                continue

    # Convert to arrays
    all_pixels = np.array(all_pixels)

    # Compute statistics
    stats = {
        'overall': {
            'mean': float(np.mean(all_pixels)),
            'std': float(np.std(all_pixels)),
            'min': float(np.min(all_pixels)),
            'max': float(np.max(all_pixels)),
            'median': float(np.median(all_pixels)),
            'p5': float(np.percentile(all_pixels, 5)),
            'p95': float(np.percentile(all_pixels, 95))
        }
    }

    for cls in classes:
        if class_pixels[cls]:
            arr = np.array(class_pixels[cls])
            stats[cls] = {
                'mean': float(np.mean(arr)),
                'std': float(np.std(arr)),
                'min': float(np.min(arr)),
                'max': float(np.max(arr)),
                'median': float(np.median(arr))
            }

    # Print statistics table
    print("\n" + "=" * 60)
    print("PIXEL STATISTICS")
    print("=" * 60)
    rows = []
    for key in ['overall'] + classes:
        if key in stats:
            rows.append({
                'Group': key.capitalize(),
                'Mean': f"{stats[key]['mean']:.2f}",
                'Std': f"{stats[key]['std']:.2f}",
                'Min': f"{stats[key]['min']:.2f}",
                'Max': f"{stats[key]['max']:.2f}",
                'Median': f"{stats[key]['median']:.2f}"
            })
    stats_df = pd.DataFrame(rows)
    print(stats_df.to_string(index=False))

    # --- Visualizations ---
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Overall pixel intensity histogram
    axes[0, 0].hist(all_pixels, bins=256, range=(0, 255), color='steelblue', edgecolor='black', alpha=0.75)
    axes[0, 0].axvline(stats['overall']['mean'], color='red', linestyle='--', linewidth=2, label=f"Mean={stats['overall']['mean']:.1f}")
    axes[0, 0].axvline(stats['overall']['median'], color='green', linestyle='--', linewidth=2, label=f"Median={stats['overall']['median']:.1f}")
    axes[0, 0].set_title('Overall Pixel Intensity Distribution', fontweight='bold')
    axes[0, 0].set_xlabel('Pixel Intensity (0-255)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    # Per-class histograms
    colors = {'normal': '#2ecc71', 'benign': '#f39c12', 'malignant': '#e74c3c'}
    for cls in classes:
        if class_pixels[cls]:
            arr = np.array(class_pixels[cls])
            axes[0, 1].hist(arr, bins=128, range=(0, 255), alpha=0.5,
                           label=cls.capitalize(), color=colors[cls], density=True)
    axes[0, 1].set_title('Pixel Intensity by Class', fontweight='bold')
    axes[0, 1].set_xlabel('Pixel Intensity (0-255)')
    axes[0, 1].set_ylabel('Density')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    # Brightness (mean per image) distribution
    brightness_data = {}
    for cls in classes:
        class_dir = dataset_dir / cls
        if not class_dir.exists():
            matches = list(dataset_dir.rglob(cls))
            class_dir = matches[0] if matches else None
        if class_dir is None:
            continue
        image_files = [f for f in class_dir.iterdir()
                       if f.suffix.lower() in {'.png', '.jpg', '.jpeg'} and '_mask' not in f.stem]
        if len(image_files) > 200:
            random.seed(42)
            image_files = random.sample(image_files, 200)
        means = []
        for img_path in image_files:
            try:
                img = np.array(Image.open(img_path).convert('L'), dtype=np.float32)
                means.append(np.mean(img))
            except:
                continue
        brightness_data[cls] = means

    for cls in classes:
        if cls in brightness_data:
            axes[1, 0].hist(brightness_data[cls], bins=30, alpha=0.6,
                           label=cls.capitalize(), color=colors[cls], edgecolor='black')
    axes[1, 0].set_title('Brightness Distribution (Mean per Image)', fontweight='bold')
    axes[1, 0].set_xlabel('Mean Pixel Value')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    # Contrast (std per image) distribution
    contrast_data = {}
    for cls in classes:
        class_dir = dataset_dir / cls
        if not class_dir.exists():
            matches = list(dataset_dir.rglob(cls))
            class_dir = matches[0] if matches else None
        if class_dir is None:
            continue
        image_files = [f for f in class_dir.iterdir()
                       if f.suffix.lower() in {'.png', '.jpg', '.jpeg'} and '_mask' not in f.stem]
        if len(image_files) > 200:
            random.seed(42)
            image_files = random.sample(image_files, 200)
        stds = []
        for img_path in image_files:
            try:
                img = np.array(Image.open(img_path).convert('L'), dtype=np.float32)
                stds.append(np.std(img))
            except:
                continue
        contrast_data[cls] = stds

    for cls in classes:
        if cls in contrast_data:
            axes[1, 1].hist(contrast_data[cls], bins=30, alpha=0.6,
                           label=cls.capitalize(), color=colors[cls], edgecolor='black')
    axes[1, 1].set_title('Contrast Distribution (Std per Image)', fontweight='bold')
    axes[1, 1].set_xlabel('Standard Deviation of Pixels')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'pixel_statistics.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Normalization note
    print("\n" + "=" * 60)
    print("NORMALIZATION STRATEGY")
    print("=" * 60)
    print(f"Dataset Mean: {stats['overall']['mean']:.4f}")
    print(f"Dataset Std:  {stats['overall']['std']:.4f}")
    print("\nThese values will be used for normalization during training:")
    print(f"  normalized = (pixel - {stats['overall']['mean']:.4f}) / {stats['overall']['std']:.4f}")
    print("\nNote: For 3-channel images, the same mean/std will be applied to all channels.")

    return stats

# Run pixel statistics analysis
pixel_stats = analyze_pixel_statistics(dataset_root)



## 4.4 Image Quality Analysis

Image quality directly impacts model performance. We analyze:
- **Blur** — Measured using Variance of Laplacian (lower = more blurry)
- **Brightness** — Mean pixel intensity (identifies overly dark/bright images)
- **Contrast** — Standard deviation of pixel values (low = washed out)
- **Noise** — Estimated using signal-to-noise proxies

Poor quality images may need exclusion or special preprocessing.


In [ ]:

# ============================================================
# 4.4 Image Quality Analysis
# ============================================================

def analyze_image_quality(dataset_dir: Path, sample_per_class: int = 200) -> pd.DataFrame:
    """
    Analyze image quality metrics: blur, brightness, contrast, noise.

    Args:
        dataset_dir: Path to dataset root.
        sample_per_class: Number of images to sample per class.

    Returns:
        DataFrame with quality metrics for each image.
    """
    classes = ["normal", "benign", "malignant"]
    quality_data = []

    print("Analyzing image quality...")

    for cls in classes:
        class_dir = dataset_dir / cls
        if not class_dir.exists():
            matches = list(dataset_dir.rglob(cls))
            class_dir = matches[0] if matches else None
        if class_dir is None:
            continue

        image_files = [f for f in class_dir.iterdir()
                       if f.suffix.lower() in {'.png', '.jpg', '.jpeg'} and '_mask' not in f.stem]

        if len(image_files) > sample_per_class:
            random.seed(42)
            image_files = random.sample(image_files, sample_per_class)

        for img_path in tqdm(image_files, desc=f"Quality check {cls}", leave=False):
            try:
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue

                # Blur: Variance of Laplacian
                laplacian_var = cv2.Laplacian(img, cv2.CV_64F).var()

                # Brightness: mean pixel value
                brightness = np.mean(img)

                # Contrast: standard deviation
                contrast = np.std(img)

                # Noise estimate: using median absolute deviation
                median = np.median(img)
                mad = np.median(np.abs(img - median))
                noise_estimate = mad * 1.4826  # Convert to std estimate

                quality_data.append({
                    'class': cls,
                    'filename': img_path.name,
                    'blur': laplacian_var,
                    'brightness': brightness,
                    'contrast': contrast,
                    'noise': noise_estimate,
                    'snr': contrast / (noise_estimate + 1e-6)  # Signal-to-noise ratio
                })
            except Exception as e:
                continue

    df = pd.DataFrame(quality_data)

    # --- Summary Statistics ---
    print("\n" + "=" * 60)
    print("IMAGE QUALITY SUMMARY")
    print("=" * 60)
    summary = df.groupby('class')[['blur', 'brightness', 'contrast', 'noise', 'snr']].agg(['mean', 'std']).round(2)
    print(summary)

    # --- Detect outliers ---
    print("\n--- QUALITY OUTLIERS ---")

    # Very blurry images (bottom 5% of Laplacian variance)
    blur_threshold = df['blur'].quantile(0.05)
    blurry = df[df['blur'] < blur_threshold]
    print(f"Very blurry images (Laplacian var < {blur_threshold:.1f}): {len(blurry)}")

    # Extremely dark images (bottom 2%)
    dark_threshold = df['brightness'].quantile(0.02)
    dark = df[df['brightness'] < dark_threshold]
    print(f"Extremely dark images (brightness < {dark_threshold:.1f}): {len(dark)}")

    # Extremely bright images (top 2%)
    bright_threshold = df['brightness'].quantile(0.98)
    bright = df[df['brightness'] > bright_threshold]
    print(f"Extremely bright images (brightness > {bright_threshold:.1f}): {len(bright)}")

    # --- Visualizations ---
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    colors = {'normal': '#2ecc71', 'benign': '#f39c12', 'malignant': '#e74c3c'}

    metrics = ['blur', 'brightness', 'contrast', 'noise', 'snr']
    positions = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1)]

    for metric, (row, col) in zip(metrics, positions):
        for cls in classes:
            cls_data = df[df['class'] == cls][metric]
            axes[row, col].hist(cls_data, bins=30, alpha=0.6,
                               label=cls.capitalize(), color=colors[cls], edgecolor='black')
        axes[row, col].set_title(f'{metric.capitalize()} Distribution', fontweight='bold')
        axes[row, col].set_xlabel(metric.capitalize())
        axes[row, col].set_ylabel('Frequency')
        axes[row, col].legend()
        axes[row, col].grid(alpha=0.3)

    # Show outlier examples
    axes[1, 2].axis('off')
    outlier_text = (
        "QUALITY OUTLIERS DETECTED\n"
        "=" * 30 + "\n"
        f"Very blurry: {len(blurry)}\n"
        f"Too dark: {len(dark)}\n"
        f"Too bright: {len(bright)}\n"
        "\n"
        "Recommendation:\n"
        "- Blur: Data augmentation\n"
        "- Darkness: Adjust brightness\n"
        "- Brightness: Normalize intensities"
    )
    axes[1, 2].text(0.1, 0.5, outlier_text, fontsize=11, family='monospace',
                    verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'image_quality.png', dpi=150, bbox_inches='tight')
    plt.show()

    return df

# Run image quality analysis
quality_df = analyze_image_quality(dataset_root)



## 4.5 Sample Visualization

Visual inspection confirms:
- Labels are correctly assigned
- Image quality is adequate
- Classes are visually distinguishable
- Annotation consistency

We display random samples from each class with comments on visual characteristics.


In [ ]:

# ============================================================
# 4.5 Sample Visualization
# ============================================================

def visualize_samples(dataset_dir: Path, samples_per_class: int = 8) -> None:
    """
    Display random sample images from each class with their labels.

    Args:
        dataset_dir: Path to dataset root.
        samples_per_class: Number of random samples to show per class.
    """
    classes = ["normal", "benign", "malignant"]
    fig, axes = plt.subplots(len(classes), samples_per_class, figsize=(18, 8))
    fig.suptitle("Sample Images from Each Class", fontsize=16, fontweight="bold", y=0.98)

    colors = {"normal": "#2ecc71", "benign": "#f39c12", "malignant": "#e74c3c"}

    for row, cls in enumerate(classes):
        class_dir = dataset_dir / cls
        if not class_dir.exists():
            matches = list(dataset_dir.rglob(cls))
            class_dir = matches[0] if matches else None
        if class_dir is None:
            continue

        image_files = [f for f in class_dir.iterdir()
                       if f.suffix.lower() in {".png", ".jpg", ".jpeg"} and "_mask" not in f.stem]

        # Random sample
        random.seed(42 + row)
        samples = random.sample(image_files, min(samples_per_class, len(image_files)))

        for col, img_path in enumerate(samples):
            img = Image.open(img_path).convert("RGB")
            axes[row, col].imshow(img)
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(
                    cls.capitalize(),
                    fontsize=13, fontweight="bold", color=colors[cls],
                    rotation=0, ha="right", va="center", labelpad=20
                )

    plt.tight_layout(rect=[0.02, 0, 1, 0.95])
    plt.savefig(RESULTS_DIR / "sample_images.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Print visual characteristics
    print("\n" + "=" * 60)
    print("VISUAL CHARACTERISTICS BY CLASS")
    print("=" * 60)
    print("NORMAL:")
    print("  - Homogeneous, uniform texture")
    print("  - No distinct masses or irregularities")
    print("  - Consistent gray-scale appearance")
    print("")
    print("BENIGN:")
    print("  - Well-defined, smooth boundaries")
    print("  - Oval or round shapes")
    print("  - Hypoechoic (darker) regions")
    print("")
    print("MALIGNANT:")
    print("  - Irregular, spiculated boundaries")
    print("  - Ill-defined margins")
    print("  - Heterogeneous internal texture")

visualize_samples(dataset_root)



## 4.6 Segmentation Mask Analysis

BUSI provides expert-annotated segmentation masks for benign and malignant lesions.
Although our current task is **classification**, masks are valuable for:
- **Explainability** — Highlighting regions the model should focus on
- **Future work** — Training a segmentation model (multi-task learning)
- **Quality control** — Verifying that annotations are consistent

We analyze mask properties: area, size distribution, and bounding boxes.


In [ ]:

# ============================================================
# 4.6 Segmentation Mask Analysis
# ============================================================

def analyze_masks(dataset_dir: Path) -> pd.DataFrame:
    """
    Analyze segmentation masks: count, area, bounding boxes.

    Args:
        dataset_dir: Path to dataset root.

    Returns:
        DataFrame with mask statistics.
    """
    classes = ["normal", "benign", "malignant"]
    mask_data = []
    overlay_examples = []

    print("Analyzing segmentation masks...")

    for cls in classes:
        class_dir = dataset_dir / cls
        if not class_dir.exists():
            matches = list(dataset_dir.rglob(cls))
            class_dir = matches[0] if matches else None
        if class_dir is None:
            continue

        # Get all mask files
        mask_files = sorted([f for f in class_dir.iterdir()
                            if f.suffix.lower() in {".png", ".jpg", ".jpeg"} and "_mask" in f.stem])

        print(f"{cls}: {len(mask_files)} masks found")

        for mask_path in tqdm(mask_files, desc=f"Masks {cls}", leave=False):
            try:
                mask = np.array(Image.open(mask_path).convert("L"))
                h, w = mask.shape
                total_pixels = h * w

                # Binary mask (any non-zero is lesion)
                binary_mask = (mask > 0).astype(np.uint8)
                lesion_pixels = np.sum(binary_mask)
                lesion_area_pct = (lesion_pixels / total_pixels) * 100

                # Bounding box
                coords = np.where(binary_mask > 0)
                if len(coords[0]) > 0:
                    bbox_y_min, bbox_y_max = coords[0].min(), coords[0].max()
                    bbox_x_min, bbox_x_max = coords[1].min(), coords[1].max()
                    bbox_w = bbox_x_max - bbox_x_min + 1
                    bbox_h = bbox_y_max - bbox_y_min + 1
                else:
                    bbox_x_min = bbox_y_min = bbox_w = bbox_h = 0

                mask_data.append({
                    "class": cls,
                    "mask_file": mask_path.name,
                    "width": w,
                    "height": h,
                    "lesion_pixels": int(lesion_pixels),
                    "lesion_area_pct": lesion_area_pct,
                    "bbox_x": bbox_x_min,
                    "bbox_y": bbox_y_min,
                    "bbox_w": bbox_w,
                    "bbox_h": bbox_h
                })

                # Save a few overlay examples
                if len(overlay_examples) < 6 and cls in ["benign", "malignant"]:
                    img_name = mask_path.stem.replace("_mask", "")
                    img_candidates = list(class_dir.glob(f"{img_name}.*"))
                    img_candidates = [c for c in img_candidates if "_mask" not in c.stem]
                    if img_candidates:
                        overlay_examples.append({
                            "class": cls,
                            "image_path": img_candidates[0],
                            "mask_path": mask_path
                        })
            except Exception as e:
                continue

    df = pd.DataFrame(mask_data)

    # --- Summary ---
    print("\n" + "=" * 60)
    print("MASK STATISTICS")
    print("=" * 60)
    if not df.empty:
        summary = df.groupby("class").agg({
            "lesion_pixels": ["mean", "std"],
            "lesion_area_pct": ["mean", "std", "min", "max"],
            "bbox_w": ["mean"],
            "bbox_h": ["mean"]
        }).round(2)
        print(summary)
    else:
        print("No mask data available.")

    # --- Visualize overlays ---
    if overlay_examples:
        n_examples = min(len(overlay_examples), 6)
        fig, axes = plt.subplots(3, n_examples, figsize=(16, 8))
        fig.suptitle("Original Image | Mask | Overlay", fontsize=14, fontweight="bold")

        for i, example in enumerate(overlay_examples[:n_examples]):
            img = np.array(Image.open(example["image_path"]).convert("RGB"))
            mask = np.array(Image.open(example["mask_path"]).convert("L"))
            binary_mask = (mask > 0).astype(np.uint8)

            # Create overlay (red mask on image)
            overlay = img.copy()
            overlay[binary_mask > 0] = [255, 0, 0]  # Red overlay
            overlay = cv2.addWeighted(img, 0.7, overlay, 0.3, 0)

            axes[0, i].imshow(img)
            axes[0, i].set_title(example["class"].capitalize(), fontsize=10)
            axes[0, i].axis("off")

            axes[1, i].imshow(mask, cmap="gray")
            axes[1, i].axis("off")

            axes[2, i].imshow(overlay)
            axes[2, i].axis("off")

        # Row labels
        axes[0, 0].set_ylabel("Original", fontsize=11, fontweight="bold", rotation=0, ha="right", va="center", labelpad=20)
        axes[1, 0].set_ylabel("Mask", fontsize=11, fontweight="bold", rotation=0, ha="right", va="center", labelpad=20)
        axes[2, 0].set_ylabel("Overlay", fontsize=11, fontweight="bold", rotation=0, ha="right", va="center", labelpad=20)

        plt.tight_layout(rect=[0, 0, 1, 0.94])
        plt.savefig(RESULTS_DIR / "mask_overlays.png", dpi=150, bbox_inches="tight")
        plt.show()

    # Distribution of lesion area
    if not df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for cls, color in zip(["benign", "malignant"], ["#f39c12", "#e74c3c"]):
            cls_data = df[df["class"] == cls]["lesion_area_pct"]
            if not cls_data.empty:
                axes[0].hist(cls_data, bins=25, alpha=0.6, label=cls.capitalize(), color=color, edgecolor="black")
        axes[0].set_title("Lesion Area Distribution (% of Image)", fontweight="bold")
        axes[0].set_xlabel("Lesion Area (%)")
        axes[0].set_ylabel("Frequency")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        for cls, color in zip(["benign", "malignant"], ["#f39c12", "#e74c3c"]):
            cls_data = df[df["class"] == cls]["lesion_pixels"]
            if not cls_data.empty:
                axes[1].hist(cls_data, bins=25, alpha=0.6, label=cls.capitalize(), color=color, edgecolor="black")
        axes[1].set_title("Lesion Size Distribution (Pixels)", fontweight="bold")
        axes[1].set_xlabel("Lesion Size (pixels)")
        axes[1].set_ylabel("Frequency")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "mask_distributions.png", dpi=150, bbox_inches="tight")
        plt.show()

    return df

mask_df = analyze_masks(dataset_root)



## 4.7 Dataset Summary

Final comprehensive report summarizing all EDA findings, strengths, weaknesses, and recommendations for preprocessing and augmentation.


In [ ]:

# ============================================================
# 4.7 Dataset Summary Report
# ============================================================

def generate_eda_summary():
    """Generate a comprehensive EDA summary report."""

    report = """
====================================================================
                 BUSI DATASET EDA SUMMARY REPORT
====================================================================

1. CLASS DISTRIBUTION
   -------------------
   - The dataset has 3 classes: Normal, Benign, Malignant
   - Class distribution may be imbalanced
   - Recommendation: Use class weighting or stratified sampling

2. IMAGE STATISTICS
   -----------------
   - Images vary in resolution
   - All images will be resized to 224x224 (standard for CNNs)
   - Most images are grayscale (will be converted to 3-channel RGB)
   - Aspect ratios vary; resizing will standardize them

3. PIXEL STATISTICS
   -----------------
   - Dataset-specific mean and std computed for normalization
   - Using dataset statistics rather than ImageNet defaults
   - Normalization formula: (x - mean) / std

4. IMAGE QUALITY
   --------------
   - Some blurry images detected (low Laplacian variance)
   - Some dark/bright outliers identified
   - Overall quality is acceptable for training
   - Augmentation will help with quality variations

5. SEGMENTATION MASKS
   -------------------
   - Masks available for benign and malignant cases
   - Lesions occupy variable portions of images
   - Masks useful for future explainability features

6. DATASET STRENGTHS
   ------------------
   - Real clinical ultrasound images
   - Expert annotations with segmentation masks
   - Multiple classes representing clinical scenarios
   - Reasonable dataset size for transfer learning

7. DATASET WEAKNESSES
   -------------------
   - Potential class imbalance
   - Variable image quality
   - Variable resolutions
   - Limited total size (may need augmentation)

8. RECOMMENDED PREPROCESSING
   --------------------------
   - Resize all images to 224x224
   - Convert grayscale to RGB (3 channels)
   - Normalize with dataset-specific statistics
   - Apply data augmentation

9. RECOMMENDED AUGMENTATION
   -------------------------
   - Horizontal flip (anatomically plausible)
   - Vertical flip (less common but useful)
   - Random rotation (up to 15 degrees)
   - Random brightness/contrast adjustment
   - Random zoom/crop
   - These augmentations simulate real ultrasound variations

10. TRAINING STRATEGY
    ------------------
    - Use transfer learning with ImageNet pre-trained weights
    - Fine-tune on BUSI dataset
    - Use stratified split (70/15/15)
    - Monitor F1-score for each class
    - Apply class weighting if imbalance detected
====================================================================
"""
    print(report)

    # Save report
    report_path = RESULTS_DIR / "eda_summary_report.txt"
    with open(report_path, "w") as f:
        f.write(report)
    print(f"\nEDA report saved to: {report_path}")

generate_eda_summary()



---

# Section 5: Data Preprocessing

This section implements the complete preprocessing pipeline:

### Steps:
1. **Resize** — Standardize to 224x224 for model input
2. **Grayscale → RGB** — Convert 1-channel to 3-channel
3. **Normalize** — Using dataset-specific mean and std
4. **Data Augmentation** — Increase effective dataset size
5. **Train/Validation/Test Split** — Stratified to preserve class ratios
6. **Custom Dataset & DataLoaders** — Efficient batch loading

### Augmentation Strategy for Ultrasound:
- **Horizontal flip** — Anatomically valid (left/right breast)
- **Vertical flip** — Sometimes valid depending on probe orientation
- **Rotation** ±15° — Accounts for probe angle variations
- **Brightness/Contrast** — Simulates different machine settings
- **Random crop/zoom** — Simulates different zoom levels

All augmentations are **medically plausible** — they represent real variations in ultrasound imaging.


In [ ]:

# ============================================================
# 5.1 Configuration and Helper Functions
# ============================================================

# --- Preprocessing Configuration ---
INPUT_SIZE = 224  # Standard input size for EfficientNet
BATCH_SIZE = 32   # Adjust based on GPU memory
NUM_WORKERS = 2   # DataLoader workers (2 for Colab)

# Normalization values (will be computed from dataset or use defaults)
# These will be updated after computing dataset statistics
NORMALIZE_MEAN = [0.5, 0.5, 0.5]   # Placeholder, computed during EDA
NORMALIZE_STD = [0.5, 0.5, 0.5]    # Placeholder, computed during EDA

# --- Split Configuration ---
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Verify ratios sum to 1.0
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, "Split ratios must sum to 1.0"

print("Preprocessing Configuration:")
print(f"  Input size:      {INPUT_SIZE}x{INPUT_SIZE}")
print(f"  Batch size:      {BATCH_SIZE}")
print(f"  Train ratio:     {TRAIN_RATIO}")
print(f"  Val ratio:       {VAL_RATIO}")
print(f"  Test ratio:      {TEST_RATIO}")


In [ ]:

# ============================================================
# 5.2 Dataset Splitting (Stratified)
# ============================================================
# We use stratified splitting to ensure each set has the same
# class proportions as the full dataset. This is critical for
# imbalanced medical datasets.

def prepare_dataset_splits(dataset_dir: Path):
    """
    Prepare stratified train/validation/test splits.

    Args:
        dataset_dir: Path to dataset root containing class folders.

    Returns:
        Dictionary with 'train', 'val', 'test' keys containing
        lists of (image_path, class_index) tuples.
    """
    classes = ["normal", "benign", "malignant"]
    all_files = []
    all_labels = []

    print("Collecting dataset files...")

    for idx, cls in enumerate(classes):
        class_dir = dataset_dir / cls
        if not class_dir.exists():
            matches = list(dataset_dir.rglob(cls))
            class_dir = matches[0] if matches else None

        if class_dir is None or not class_dir.exists():
            print(f"Warning: Class directory '{cls}' not found.")
            continue

        image_files = [f for f in class_dir.iterdir()
                       if f.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}
                       and "_mask" not in f.stem]

        for img_path in image_files:
            all_files.append(str(img_path))
            all_labels.append(idx)

    print(f"Total images collected: {len(all_files)}")

    # First split: separate test set (15%)
    train_val_files, test_files, train_val_labels, test_labels = train_test_split(
        all_files, all_labels,
        test_size=TEST_RATIO,
        random_state=SEED,
        stratify=all_labels
    )

    # Second split: separate validation from train (15% of total = 17.6% of remaining)
    val_ratio_adjusted = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
    train_files, val_files, train_labels, val_labels = train_test_split(
        train_val_files, train_val_labels,
        test_size=val_ratio_adjusted,
        random_state=SEED,
        stratify=train_val_labels
    )

    splits = {
        "train": list(zip(train_files, train_labels)),
        "val": list(zip(val_files, val_labels)),
        "test": list(zip(test_files, test_labels))
    }

    # Print split statistics
    print("\n" + "=" * 60)
    print("DATASET SPLITS")
    print("=" * 60)
    for split_name, split_data in splits.items():
        labels = [label for _, label in split_data]
        counts = Counter(labels)
        print(f"\n{split_name.upper()}:")
        print(f"  Total: {len(split_data)}")
        for idx, cls in enumerate(classes):
            pct = (counts.get(idx, 0) / len(split_data)) * 100 if split_data else 0
            print(f"    {cls:12s}: {counts.get(idx, 0):4d} ({pct:5.1f}%)")

    return splits

# Prepare splits
dataset_root = LOCAL_DATASET_DIR / "Dataset" if (LOCAL_DATASET_DIR / "Dataset").exists() else LOCAL_DATASET_DIR
splits = prepare_dataset_splits(dataset_root)


In [ ]:

# ============================================================
# 5.3 Data Transforms
# ============================================================
# Define separate transforms for training (with augmentation)
# and validation/testing (without augmentation).

# Compute dataset mean and std for normalization
# Using the pixel_stats from EDA section if available
try:
    # Use computed statistics from EDA
    dataset_mean = pixel_stats["overall"]["mean"] / 255.0
    dataset_std = pixel_stats["overall"]["std"] / 255.0
    NORMALIZE_MEAN = [dataset_mean] * 3
    NORMALIZE_STD = [dataset_std] * 3
    print(f"Using computed dataset statistics:")
except NameError:
    # Fallback: use typical ultrasound values
    NORMALIZE_MEAN = [0.485, 0.456, 0.406]  # ImageNet defaults as fallback
    NORMALIZE_STD = [0.229, 0.224, 0.225]
    print(f"Using ImageNet normalization (fallback):")

print(f"  Mean: {NORMALIZE_MEAN}")
print(f"  Std:  {NORMALIZE_STD}")

# --- Training Transforms (with augmentation) ---
train_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.Grayscale(num_output_channels=3),  # Convert to 3-channel
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.RandomResizedCrop(size=INPUT_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
])

# --- Validation/Test Transforms (no augmentation) ---
val_test_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.Grayscale(num_output_channels=3),  # Convert to 3-channel
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
])

print("\nTransforms defined:")
print("  Train: Resize -> Grayscale(3ch) -> Augmentation -> Normalize")
print("  Val/Test: Resize -> Grayscale(3ch) -> Normalize")


In [ ]:

# ============================================================
# 5.4 Custom PyTorch Dataset Class
# ============================================================

class BUSIDataset(Dataset):
    """
    Custom PyTorch Dataset for the BUSI breast ultrasound dataset.

    Args:
        data: List of (image_path, label) tuples.
        transform: torchvision transforms to apply.
        preload: If True, preload all images into memory (faster but more RAM).
    """

    def __init__(self, data: List[Tuple[str, int]], transform=None, preload: bool = False):
        self.data = data
        self.transform = transform
        self.preload = preload
        self.images = {}

        if preload:
            print("Preloading images into memory...")
            for img_path, _ in tqdm(data, desc="Loading"):
                try:
                    self.images[img_path] = Image.open(img_path).convert("RGB")
                except Exception as e:
                    print(f"Error loading {img_path}: {e}")

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img_path, label = self.data[idx]

        if self.preload and img_path in self.images:
            image = self.images[img_path].copy()
        else:
            try:
                image = Image.open(img_path).convert("RGB")
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
                # Return a blank image as fallback
                image = Image.new("RGB", (INPUT_SIZE, INPUT_SIZE), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        return image, label


def create_dataloaders(splits: Dict, batch_size: int = 32, num_workers: int = 2):
    """
    Create DataLoaders for train, validation, and test sets.

    Args:
        splits: Dictionary with 'train', 'val', 'test' splits.
        batch_size: Batch size for all loaders.
        num_workers: Number of worker processes.

    Returns:
        Dictionary of DataLoaders.
    """
    # Create datasets
    train_dataset = BUSIDataset(splits["train"], transform=train_transforms)
    val_dataset = BUSIDataset(splits["val"], transform=val_test_transforms)
    test_dataset = BUSIDataset(splits["test"], transform=val_test_transforms)

    # Compute class weights for handling imbalance
    train_labels = [label for _, label in splits["train"]]
    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    sample_weights = [class_weights[label] for label in train_labels]

    # Create weighted sampler for training
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if num_workers > 0 else False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if num_workers > 0 else False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if num_workers > 0 else False
    )

    loaders = {
        "train": train_loader,
        "val": val_loader,
        "test": test_loader
    }

    # Print loader info
    print("\n" + "=" * 60)
    print("DATALOADERS CREATED")
    print("=" * 60)
    for name, loader in loaders.items():
        print(f"{name:5s}: {len(loader)} batches x {batch_size} = ~{len(loader.dataset)} samples")

    print(f"\nClass weights for imbalance handling:")
    for idx, w in enumerate(class_weights):
        print(f"  {CLASS_NAMES[idx]}: {w:.4f}")

    return loaders, class_weights

# Create DataLoaders
dataloaders, class_weights_array = create_dataloaders(splits, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)



---

# Section 6: Model Selection

We compare four CNN architectures commonly used for medical image classification:

| Model | Parameters | Strengths | Weaknesses |
|-------|-----------|-----------|------------|
| **EfficientNet-B0** | ~5.3M | Best accuracy/efficiency tradeoff, compound scaling | Slightly slower than MobileNet |
| **ResNet18** | ~11.7M | Simple, proven, easy to modify | Lower accuracy than newer models |
| **DenseNet121** | ~8.0M | Feature reuse, strong gradients | Memory-intensive |
| **MobileNetV3** | ~5.4M | Fastest inference, mobile-optimized | Slightly lower accuracy |

### Why EfficientNet-B0 for Ultrasound?
1. **Compound scaling** — Balances depth, width, and resolution optimally
2. **State-of-the-art efficiency** — Best accuracy per parameter count
3. **Transfer learning friendly** — Pretrained on ImageNet, fine-tunes well
4. **Medical imaging proven** — Widely used in radiology applications
5. **Fast enough** — ~5M parameters train quickly on Colab GPU

### Fallback Strategy:
If EfficientNet is unavailable in the installed torchvision version, we fall back to **ResNet18**, which is universally available and well-tested.


In [ ]:

# ============================================================
# 6.1 Model Comparison Table
# ============================================================

def compare_models():
    """
    Compare model architectures for breast ultrasound classification.
    Displays a comparison table and parameter counts.
    """
    models_info = [
        {
            "Model": "EfficientNet-B0",
            "Parameters": "5.3M",
            "Top-1 Acc": "77.1%",
            "Size": "21 MB",
            "Speed": "Fast",
            "Best For": "Accuracy/Efficiency balance"
        },
        {
            "Model": "ResNet18",
            "Parameters": "11.7M",
            "Top-1 Acc": "69.8%",
            "Size": "47 MB",
            "Speed": "Very Fast",
            "Best For": "Simplicity, availability"
        },
        {
            "Model": "DenseNet121",
            "Parameters": "8.0M",
            "Top-1 Acc": "74.6%",
            "Size": "32 MB",
            "Speed": "Moderate",
            "Best For": "Feature reuse"
        },
        {
            "Model": "MobileNetV3-Small",
            "Parameters": "2.5M",
            "Top-1 Acc": "67.5%",
            "Size": "10 MB",
            "Speed": "Fastest",
            "Best For": "Edge/mobile deployment"
        }
    ]

    df = pd.DataFrame(models_info)
    print("=" * 80)
    print("MODEL COMPARISON FOR BREAST ULTRASOUND CLASSIFICATION")
    print("=" * 80)
    print(df.to_string(index=False))

    print("\n" + "=" * 80)
    print("RECOMMENDATION: EfficientNet-B0")
    print("=" * 80)
    print("""
EfficientNet-B0 offers the best combination of:
- High accuracy (best ImageNet top-1 accuracy among compared models)
- Low parameter count (5.3M — trains quickly on Colab)
- Compound scaling (optimal depth/width/resolution balance)
- Strong transfer learning performance on medical images
- Fast inference for real-time clinical use

ResNet18 is kept as a fallback if EfficientNet is unavailable.
    """)

    return df

model_comparison_df = compare_models()



---

# Section 7: Build the Classification Model

We use **transfer learning** with ImageNet pre-trained weights:

1. **Load pretrained backbone** — EfficientNet-B0 trained on ImageNet
2. **Freeze backbone initially** — Preserve low-level feature detectors (edges, textures)
3. **Replace classifier head** — New fully-connected layer for 3 classes
4. **Progressive unfreezing** (optional) — Fine-tune deeper layers later

### Architectural Modifications:
- Original EfficientNet-B0 outputs 1000 classes (ImageNet)
- We replace the final `Linear(1280, 1000)` with `Linear(1280, 3)`
- Added **Dropout(0.3)** for regularization (medical datasets are small)

### Why Freeze Initially?
- Prevents overfitting on small medical datasets
- Preserves general low-level features (edges, blobs, textures)
- Only trains the classifier head first
- After initial convergence, selective layers can be unfrozen


In [ ]:

# ============================================================
# 7.1 Model Architecture Definition
# ============================================================

class BreastUltrasoundClassifier(nn.Module):
    """
    Breast Ultrasound Classification Model based on EfficientNet-B0.

    Uses transfer learning with a custom classification head for 3 classes:
    Normal, Benign, Malignant.

    Args:
        num_classes: Number of output classes (default: 3).
        dropout_rate: Dropout probability for regularization.
        pretrained: Whether to use ImageNet pretrained weights.
    """

    def __init__(self, num_classes: int = 3, dropout_rate: float = 0.3, pretrained: bool = True):
        super(BreastUltrasoundClassifier, self).__init__()

        # Load pretrained EfficientNet-B0
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.efficientnet_b0(weights=weights)

        # Get the number of features from the original classifier
        in_features = self.backbone.classifier[1].in_features  # 1280

        # Replace classifier head:
        # Original: Sequential(Dropout, Linear(1280, 1000))
        # New:      Sequential(Dropout, Linear(1280, 3))
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, num_classes)
        )

        print(f"Model created with EfficientNet-B0 backbone")
        print(f"  Input features: {in_features}")
        print(f"  Output classes: {num_classes}")
        print(f"  Dropout rate:   {dropout_rate}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass through the network."""
        return self.backbone(x)

    def freeze_backbone(self):
        """Freeze all backbone parameters (only train classifier)."""
        for param in self.backbone.features.parameters():
            param.requires_grad = False
        print("Backbone frozen. Only classifier head will be trained.")

    def unfreeze_backbone(self, lr_multiplier: float = 0.1):
        """
        Unfreeze backbone for fine-tuning.

        Args:
            lr_multiplier: Learning rate multiplier for backbone layers.
        """
        for param in self.backbone.features.parameters():
            param.requires_grad = True
        print(f"Backbone unfrozen for fine-tuning (lr_multiplier={lr_multiplier}).")

    def get_trainable_params(self) -> int:
        """Count the number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_total_params(self) -> int:
        """Count total parameters (trainable + frozen)."""
        return sum(p.numel() for p in self.parameters())


# ============================================================
# Model Instantiation
# ============================================================
model = BreastUltrasoundClassifier(num_classes=NUM_CLASSES, dropout_rate=0.3, pretrained=True)
model = model.to(device)

# Freeze backbone for initial training
model.freeze_backbone()

# Print model summary
total_params = model.get_total_params()
trainable_params = model.get_trainable_params()

print("\n" + "=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")
print(f"Trainable ratio:      {trainable_params/total_params*100:.1f}%")
print(f"Device:               {device}")



---

# Section 8: Training Pipeline

Production-quality training implementation including:

### Components:
1. **CrossEntropyLoss** — Standard multi-class classification loss
2. **AdamW Optimizer** — Adam with weight decay decoupling
3. **Learning Rate Scheduler** — Cosine annealing for smooth convergence
4. **Mixed Precision Training** — Faster training with reduced memory (if GPU supports)
5. **Validation Loop** — Evaluate after every epoch
6. **Early Stopping** — Prevent overfitting
7. **Best Model Checkpointing** — Save the best model to Google Drive
8. **Progress Bars** — Real-time training progress with tqdm

### Training Strategy:
- **Phase 1:** Train only the classifier head (frozen backbone) — ~10 epochs
- **Phase 2:** Fine-tune entire network (unfrozen backbone) — ~10-20 epochs
- **Class weighting** — Handle any class imbalance


In [ ]:

# ============================================================
# 8.1 Training Configuration
# ============================================================

class TrainingConfig:
    """Configuration for model training."""
    # Training
    NUM_EPOCHS_PHASE1 = 10      # Train classifier head only
    NUM_EPOCHS_PHASE2 = 20      # Fine-tune entire network
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    EARLY_STOPPING_PATIENCE = 7

    # Mixed Precision
    USE_AMP = torch.cuda.is_available()  # Enable if GPU available

    # Checkpointing
    CHECKPOINT_DIR = CHECKPOINT_DIR
    BEST_MODEL_NAME = "classification_model.pth"

    # Logging
    PRINT_FREQ = 1  # Print every N batches

config = TrainingConfig()

print("Training Configuration:")
print(f"  Phase 1 epochs: {config.NUM_EPOCHS_PHASE1} (frozen backbone)")
print(f"  Phase 2 epochs: {config.NUM_EPOCHS_PHASE2} (unfrozen backbone)")
print(f"  Learning rate:  {config.LEARNING_RATE}")
print(f"  Weight decay:   {config.WEIGHT_DECAY}")
print(f"  Early stopping: {config.EARLY_STOPPING_PATIENCE} epochs patience")
print(f"  Mixed precision:{config.USE_AMP}")


In [ ]:

# ============================================================
# 8.2 Loss Function and Optimizer Setup
# ============================================================

# --- Loss Function with Class Weighting ---
# Convert class weights to tensor
class_weights_tensor = torch.tensor(class_weights_array, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
print(f"Loss function: CrossEntropyLoss with class weights: {class_weights_array}")
print(f"Label smoothing: 0.1 (helps prevent overconfidence)")

# --- Optimizer ---
# Start with classifier head parameters only (backbone is frozen)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)
print(f"Optimizer: AdamW (lr={config.LEARNING_RATE}, weight_decay={config.WEIGHT_DECAY})")

# --- Learning Rate Scheduler ---
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.NUM_EPOCHS_PHASE1 + config.NUM_EPOCHS_PHASE2,
    eta_min=1e-6
)
print("Scheduler: CosineAnnealingLR (smooth LR decay)")

# --- Mixed Precision Scaler ---
scaler = GradScaler() if config.USE_AMP else None
if config.USE_AMP:
    print("Mixed Precision Training (AMP) enabled")
else:
    print("Mixed Precision Training disabled (CPU mode)")


In [ ]:

# ============================================================
# 8.3 Training and Validation Functions
# ============================================================

def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    scaler: GradScaler = None,
    epoch: int = 0
) -> Tuple[float, float]:
    """
    Train the model for one epoch.

    Args:
        model: The neural network model.
        dataloader: Training data loader.
        criterion: Loss function.
        optimizer: Optimizer instance.
        device: Computation device (cuda/cpu).
        scaler: Gradient scaler for mixed precision.
        epoch: Current epoch number.

    Returns:
        Tuple of (average_loss, accuracy).
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Train Epoch {epoch}", leave=False)

    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass with mixed precision
        if scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Backward pass
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{100.*correct/total:.1f}%"
        })

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc


def validate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    epoch: int = 0
) -> Tuple[float, float]:
    """
    Validate the model on the validation set.

    Args:
        model: The neural network model.
        dataloader: Validation data loader.
        criterion: Loss function.
        device: Computation device.
        epoch: Current epoch number.

    Returns:
        Tuple of (average_loss, accuracy).
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    pbar = tqdm(dataloader, desc=f"Val Epoch {epoch}", leave=False)

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100.*correct/total:.1f}%"
            })

    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc


In [ ]:

# ============================================================
# 8.4 Complete Training Loop with Early Stopping & Checkpointing
# ============================================================

def train_model(
    model: nn.Module,
    dataloaders: Dict,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scheduler: optim.lr_scheduler._LRScheduler,
    device: torch.device,
    config: TrainingConfig,
    scaler: GradScaler = None,
    phase_name: str = "Phase 1"
) -> Dict:
    """
    Complete training loop with early stopping and checkpointing.

    Args:
        model: Neural network model.
        dataloaders: Dictionary with 'train' and 'val' loaders.
        criterion: Loss function.
        optimizer: Optimizer.
        scheduler: Learning rate scheduler.
        device: Computation device.
        config: Training configuration.
        scaler: Mixed precision scaler.
        phase_name: Name of training phase (for logging).

    Returns:
        Dictionary containing training history.
    """
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "learning_rates": []
    }

    best_val_acc = 0.0
    epochs_no_improve = 0
    num_epochs = config.NUM_EPOCHS_PHASE1 if "1" in phase_name else config.NUM_EPOCHS_PHASE2

    print("\n" + "=" * 60)
    print(f"STARTING {phase_name.upper()}")
    print("=" * 60)

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()
        current_lr = optimizer.param_groups[0]["lr"]

        # Training
        train_loss, train_acc = train_one_epoch(
            model, dataloaders["train"], criterion, optimizer, device, scaler, epoch
        )

        # Validation
        val_loss, val_acc = validate(
            model, dataloaders["val"], criterion, device, epoch
        )

        # Scheduler step
        if scheduler is not None:
            scheduler.step()

        # Record history
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["learning_rates"].append(current_lr)

        epoch_time = time.time() - start_time

        # Print progress
        print(f"Epoch [{epoch:2d}/{num_epochs}] | "
              f"LR: {current_lr:.6f} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
              f"Time: {epoch_time:.1f}s")

        # Best model checkpointing
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0

            # Save checkpoint
            checkpoint = {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc": val_acc,
                "phase": phase_name
            }
            checkpoint_path = Path(config.CHECKPOINT_DIR) / config.BEST_MODEL_NAME
            torch.save(checkpoint, checkpoint_path)
            print(f"  [SAVED] New best model (val_acc: {val_acc:.2f}%)")
        else:
            epochs_no_improve += 1

        # Early stopping
        if epochs_no_improve >= config.EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs "
                  f"(no improvement for {config.EARLY_STOPPING_PATIENCE} epochs).")
            break

    print(f"\n{phase_name} completed. Best val accuracy: {best_val_acc:.2f}%")

    return history


In [ ]:

# ============================================================
# 8.5 Execute Training — Phase 1 (Frozen Backbone)
# ============================================================
# Phase 1: Train only the classifier head with frozen backbone.
# This allows the new head to learn class-specific features
# without destroying the pretrained backbone weights.

print("=" * 60)
print("PHASE 1: TRAINING CLASSIFIER HEAD (Frozen Backbone)")
print("=" * 60)
print(f"Trainable parameters: {model.get_trainable_params():,}")

# Reset optimizer for phase 1
optimizer_p1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

scheduler_p1 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1,
    T_max=config.NUM_EPOCHS_PHASE1,
    eta_min=1e-6
)

# Train phase 1
history_p1 = train_model(
    model=model,
    dataloaders=dataloaders,
    criterion=criterion,
    optimizer=optimizer_p1,
    scheduler=scheduler_p1,
    device=device,
    config=config,
    scaler=scaler,
    phase_name="Phase 1"
)

print("\nPhase 1 training completed!")


In [ ]:

# ============================================================
# 8.6 Execute Training — Phase 2 (Fine-tuning)
# ============================================================
# Phase 2: Unfreeze the backbone and fine-tune the entire network
# with a lower learning rate to preserve pretrained features.

print("=" * 60)
print("PHASE 2: FINE-TUNING ENTIRE NETWORK")
print("=" * 60)

# Unfreeze backbone
model.unfreeze_backbone()

# Use a lower learning rate for fine-tuning
fine_tune_lr = config.LEARNING_RATE * 0.1  # 10x smaller

# Re-create optimizer with all parameters
optimizer_p2 = optim.AdamW(
    model.parameters(),
    lr=fine_tune_lr,
    weight_decay=config.WEIGHT_DECAY
)

scheduler_p2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2,
    T_max=config.NUM_EPOCHS_PHASE2,
    eta_min=1e-7
)

print(f"Trainable parameters: {model.get_trainable_params():,}")
print(f"Fine-tuning learning rate: {fine_tune_lr}")

# Load best checkpoint from phase 1
checkpoint_path = Path(config.CHECKPOINT_DIR) / config.BEST_MODEL_NAME
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best checkpoint from Phase 1 (val_acc: {checkpoint.get('val_acc', 0):.2f}%)")

# Train phase 2
history_p2 = train_model(
    model=model,
    dataloaders=dataloaders,
    criterion=criterion,
    optimizer=optimizer_p2,
    scheduler=scheduler_p2,
    device=device,
    config=config,
    scaler=scaler,
    phase_name="Phase 2"
)

print("\nPhase 2 fine-tuning completed!")


In [ ]:

# ============================================================
# 8.7 Visualize Training Curves
# ============================================================

def plot_training_curves(history_p1: Dict, history_p2: Dict, save_dir: Path):
    """
    Plot training and validation loss/accuracy curves.

    Args:
        history_p1: Phase 1 training history.
        history_p2: Phase 2 training history.
        save_dir: Directory to save plots.
    """
    # Combine histories
    train_loss = history_p1["train_loss"] + history_p2["train_loss"]
    train_acc = history_p1["train_acc"] + history_p2["train_acc"]
    val_loss = history_p1["val_loss"] + history_p2["val_loss"]
    val_acc = history_p1["val_acc"] + history_p2["val_acc"]

    epochs = range(1, len(train_loss) + 1)
    phase1_epochs = len(history_p1["train_loss"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    axes[0].plot(epochs, train_loss, "b-", label="Train Loss", linewidth=1.5)
    axes[0].plot(epochs, val_loss, "r-", label="Val Loss", linewidth=1.5)
    axes[0].axvline(phase1_epochs, color="gray", linestyle="--", alpha=0.7, label="Phase 1→2")
    axes[0].set_xlabel("Epoch", fontsize=12)
    axes[0].set_ylabel("Loss", fontsize=12)
    axes[0].set_title("Training & Validation Loss", fontsize=14, fontweight="bold")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy curves
    axes[1].plot(epochs, train_acc, "b-", label="Train Acc", linewidth=1.5)
    axes[1].plot(epochs, val_acc, "r-", label="Val Acc", linewidth=1.5)
    axes[1].axvline(phase1_epochs, color="gray", linestyle="--", alpha=0.7, label="Phase 1→2")
    axes[1].set_xlabel("Epoch", fontsize=12)
    axes[1].set_ylabel("Accuracy (%)", fontsize=12)
    axes[1].set_title("Training & Validation Accuracy", fontsize=14, fontweight="bold")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_dir / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Training curves saved to: {save_dir / 'training_curves.png'}")

# Plot training curves
plot_training_curves(history_p1, history_p2, RESULTS_DIR)



---

# Section 9: Model Evaluation

Comprehensive evaluation on the held-out test set using:
- **Accuracy** — Overall correct predictions
- **Precision** — Of predicted positive, how many are correct
- **Recall** — Of actual positive, how many were caught
- **F1-Score** — Harmonic mean of precision and recall
- **ROC-AUC** — Area under ROC curve (one-vs-rest for multi-class)
- **Confusion Matrix** — Per-class prediction breakdown
- **Classification Report** — Per-class metrics summary

### Why These Metrics for Medical Imaging?
- **Accuracy alone is insufficient** — Class imbalance can mask poor minority class performance
- **Recall (Sensitivity)** is critical — Missing malignant cases has severe consequences
- **Precision** matters too — False positives cause unnecessary anxiety and procedures
- **F1-Score** balances both concerns


In [ ]:

# ============================================================
# 9.1 Load Best Model
# ============================================================
# Load the best checkpoint saved during training

best_checkpoint_path = Path(config.CHECKPOINT_DIR) / config.BEST_MODEL_NAME

if best_checkpoint_path.exists():
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from Phase {checkpoint.get('phase', 'unknown')}")
    print(f"Validation accuracy at checkpoint: {checkpoint.get('val_acc', 0):.2f}%")
else:
    print("No checkpoint found. Using current model state.")

model = model.to(device)
model.eval()


In [ ]:

# ============================================================
# 9.2 Evaluate on Test Set
# ============================================================

def evaluate_model(model: nn.Module, dataloader: DataLoader, device: torch.device) -> Dict:
    """
    Comprehensive model evaluation.

    Args:
        model: Trained neural network.
        dataloader: Test data loader.
        device: Computation device.

    Returns:
        Dictionary with all evaluation metrics.
    """
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []
    all_paths = []

    print("Running evaluation on test set...")

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            images = images.to(device, non_blocking=True)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # --- Compute Metrics ---
    accuracy = accuracy_score(all_labels, all_preds)
    precision_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    precision_weighted = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    recall_weighted = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1_weighted = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    # Per-class metrics
    precision_per_class = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall_per_class = recall_score(all_labels, all_preds, average=None, zero_division=0)
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    # ROC-AUC (One-vs-Rest)
    try:
        roc_auc_macro = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc_macro = 0.0

    results = {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "roc_auc_macro": roc_auc_macro,
        "precision_per_class": precision_per_class,
        "recall_per_class": recall_per_class,
        "f1_per_class": f1_per_class,
        "all_probs": all_probs,
        "all_preds": all_preds,
        "all_labels": all_labels
    }

    return results

# Run evaluation
test_results = evaluate_model(model, dataloaders["test"], device)


In [ ]:

# ============================================================
# 9.3 Display Metrics
# ============================================================

def print_metrics(results: Dict):
    """Print formatted evaluation metrics."""
    print("\n" + "=" * 60)
    print("TEST SET EVALUATION RESULTS")
    print("=" * 60)

    # Overall metrics
    print("\n--- Overall Metrics ---")
    print(f"  Accuracy:           {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")
    print(f"  Precision (macro):  {results['precision_macro']:.4f}")
    print(f"  Recall (macro):     {results['recall_macro']:.4f}")
    print(f"  F1-Score (macro):   {results['f1_macro']:.4f}")
    print(f"  Precision (weighted): {results['precision_weighted']:.4f}")
    print(f"  Recall (weighted):    {results['recall_weighted']:.4f}")
    print(f"  F1-Score (weighted):  {results['f1_weighted']:.4f}")
    print(f"  ROC-AUC (macro, OvR): {results['roc_auc_macro']:.4f}")

    # Per-class metrics
    print("\n--- Per-Class Metrics ---")
    metrics_df = pd.DataFrame({
        "Class": [name.capitalize() for name in CLASS_NAMES],
        "Precision": results["precision_per_class"],
        "Recall": results["recall_per_class"],
        "F1-Score": results["f1_per_class"]
    })
    print(metrics_df.to_string(index=False))

    # Classification report
    print("\n--- Classification Report ---")
    print(classification_report(
        results["all_labels"],
        results["all_preds"],
        target_names=[name.capitalize() for name in CLASS_NAMES],
        digits=4
    ))

    return metrics_df

metrics_df = print_metrics(test_results)


In [ ]:

# ============================================================
# 9.4 Confusion Matrix Visualization
# ============================================================

def plot_confusion_matrix(results: Dict, save_dir: Path):
    """
    Plot and save the confusion matrix.

    Args:
        results: Evaluation results dictionary.
        save_dir: Directory to save the plot.
    """
    cm = confusion_matrix(results["all_labels"], results["all_preds"])

    fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=[name.capitalize() for name in CLASS_NAMES],
        yticklabels=[name.capitalize() for name in CLASS_NAMES],
        ax=ax,
        linewidths=1,
        linecolor="black",
        square=True,
        cbar_kws={"shrink": 0.8}
    )

    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title("Confusion Matrix (Test Set)", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.savefig(save_dir / "confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Also save normalized version
    cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt=".2%",
        cmap="Blues",
        xticklabels=[name.capitalize() for name in CLASS_NAMES],
        yticklabels=[name.capitalize() for name in CLASS_NAMES],
        ax=ax,
        linewidths=1,
        linecolor="black",
        square=True,
        cbar_kws={"shrink": 0.8}
    )
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title("Normalized Confusion Matrix (Test Set)", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.savefig(save_dir / "confusion_matrix_normalized.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_confusion_matrix(test_results, RESULTS_DIR)



---

# Section 10: Error Analysis

Analyze model failures to understand:
- Which classes are commonly confused
- Whether errors are due to image quality, similar appearance, or other factors
- Confidence levels on incorrect predictions
- Opportunities for improvement

### Common Error Sources in Medical Imaging:
1. **Similar tissue appearance** — Benign and malignant can look alike
2. **Poor image quality** — Blur, noise, artifacts
3. **Small lesions** — Hard to detect at low resolution
4. **Class imbalance** — Model biased toward majority classes


In [ ]:

# ============================================================
# 10.1 Error Analysis Functions
# ============================================================

def analyze_errors(model: nn.Module, dataloader: DataLoader, device: torch.device, max_display: int = 12):
    """
    Analyze and visualize model errors.

    Args:
        model: Trained model.
        dataloader: Test data loader.
        device: Computation device.
        max_display: Maximum number of error examples to display.

    Returns:
        Dictionary with error analysis data.
    """
    model.eval()

    correct_examples = []
    incorrect_examples = []

    print("Analyzing predictions...")

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Error analysis"):
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            confidences, predicted = probs.max(1)

            for i in range(images.size(0)):
                img_tensor = images[i].cpu()
                true_label = labels[i].item()
                pred_label = predicted[i].item()
                confidence = confidences[i].item()
                all_prob = probs[i].cpu().numpy()

                example = {
                    "image": img_tensor,
                    "true_label": true_label,
                    "pred_label": pred_label,
                    "confidence": confidence,
                    "all_probs": all_prob
                }

                if true_label == pred_label:
                    correct_examples.append(example)
                else:
                    incorrect_examples.append(example)

    print(f"\nTotal correct predictions:   {len(correct_examples)}")
    print(f"Total incorrect predictions: {len(incorrect_examples)}")

    return {
        "correct": correct_examples,
        "incorrect": incorrect_examples
    }

# Run error analysis
error_data = analyze_errors(model, dataloaders["test"], device)


In [ ]:

# ============================================================
# 10.2 Visualize Misclassified Examples
# ============================================================

def plot_error_examples(error_data: Dict, max_display: int = 12):
    """
    Display misclassified examples with prediction confidence.

    Args:
        error_data: Error analysis data from analyze_errors().
        max_display: Maximum number of examples to show.
    """
    incorrect = error_data["incorrect"]

    if not incorrect:
        print("No misclassifications to display!")
        return

    # Sort by confidence (highest confidence wrong predictions first)
    incorrect_sorted = sorted(incorrect, key=lambda x: x["confidence"], reverse=True)

    n_display = min(len(incorrect_sorted), max_display)
    n_cols = 4
    n_rows = (n_display + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    mean = torch.tensor(NORMALIZE_MEAN).view(3, 1, 1)
    std = torch.tensor(NORMALIZE_STD).view(3, 1, 1)

    for i in range(n_display):
        row, col = i // n_cols, i % n_cols
        ex = incorrect_sorted[i]

        # Denormalize image
        img = ex["image"] * std + mean
        img = torch.clamp(img, 0, 1)
        img_np = img.permute(1, 2, 0).numpy()

        axes[row, col].imshow(img_np)
        axes[row, col].axis("off")

        true_name = CLASS_NAMES[ex["true_label"]].capitalize()
        pred_name = CLASS_NAMES[ex["pred_label"]].capitalize()

        color = "red" if ex["pred_label"] != ex["true_label"] else "green"
        title = f"True: {true_name}\nPred: {pred_name}\nConf: {ex['confidence']:.2%}"
        axes[row, col].set_title(title, fontsize=9, color=color)

    # Hide unused subplots
    for i in range(n_display, n_rows * n_cols):
        row, col = i // n_cols, i % n_cols
        axes[row, col].axis("off")

    plt.suptitle("Misclassified Examples (sorted by confidence)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "misclassified_examples.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_error_examples(error_data, max_display=12)


In [ ]:

# ============================================================
# 10.3 Error Discussion
# ============================================================

print("=" * 60)
print("ERROR ANALYSIS DISCUSSION")
print("=" * 60)

print("""
COMMON ERROR SOURCES:

1. SIMILAR TISSUE APPEARANCE
   - Benign fibroadenomas and malignant masses can have similar
     echogenicity and shape in ultrasound
   - Recommendation: Use segmentation masks as auxiliary input
     or attention mechanisms

2. POOR IMAGE QUALITY
   - Blurry or noisy images reduce model confidence
   - Recommendation: Add quality-based filtering or enhance
     preprocessing for low-quality images

3. SMALL LESIONS
   - Very small abnormalities are harder to detect at 224x224
   - Recommendation: Use multi-scale training or higher resolution

4. CLASS IMBALANCE
   - If present, model may be biased toward majority class
   - Recommendation: Class weights are already applied; consider
     oversampling minority classes or focal loss

POTENTIAL IMPROVEMENTS:
- Ensemble multiple models (EfficientNet + ResNet + DenseNet)
- Use segmentation masks as additional input channel
- Implement attention mechanisms (CBAM, SE blocks)
- Collect more training data, especially for underperforming classes
- Use test-time augmentation (TTA) for more robust predictions
- Implement Grad-CAM for explainability and error analysis
""")



---

# Section 11: Prediction Function

Create a reusable `predict()` function that:
1. Takes an image path as input
2. Preprocesses the image (resize, normalize)
3. Runs inference with the trained model
4. Returns structured output with class, confidence, and probabilities

This function is designed for easy integration into the hackathon backend API.


In [ ]:

# ============================================================
# 11.1 Prediction Function
# ============================================================

def predict(image_path: str, model: nn.Module = None, device: torch.device = None) -> Dict:
    """
    Predict the class of a breast ultrasound image.

    Args:
        image_path: Path to the image file.
        model: Trained model (loads from checkpoint if None).
        device: Computation device (auto-detected if None).

    Returns:
        Dictionary with prediction results:
        {
            "prediction": "benign",
            "confidence": 0.95,
            "probabilities": {
                "normal": 0.02,
                "benign": 0.95,
                "malignant": 0.03
            }
        }
    """
    # Auto-detect device
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model if not provided
    if model is None:
        model = BreastUltrasoundClassifier(num_classes=NUM_CLASSES, pretrained=False)
        checkpoint_path = Path(config.CHECKPOINT_DIR) / config.BEST_MODEL_NAME
        if checkpoint_path.exists():
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint["model_state_dict"])
        else:
            raise FileNotFoundError(f"No checkpoint found at {checkpoint_path}")
        model = model.to(device)

    model.eval()

    # Preprocess image
    preprocess = transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
    ])

    image = Image.open(image_path).convert("RGB")
    input_tensor = preprocess(image).unsqueeze(0).to(device)

    # Inference
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)

    # Convert to numpy
    probs = probabilities[0].cpu().numpy()
    pred_idx = int(np.argmax(probs))
    confidence = float(probs[pred_idx])

    result = {
        "prediction": CLASS_NAMES[pred_idx],
        "confidence": round(confidence, 4),
        "probabilities": {
            name: round(float(prob), 4) for name, prob in zip(CLASS_NAMES, probs)
        }
    }

    return result


def visualize_prediction(image_path: str, result: Dict):
    """
    Display the image with prediction overlay.

    Args:
        image_path: Path to the image.
        result: Prediction result dictionary.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Show image
    img = Image.open(image_path).convert("RGB")
    axes[0].imshow(img)
    axes[0].axis("off")
    axes[0].set_title("Input Image", fontsize=12, fontweight="bold")

    # Show prediction probabilities
    classes = [name.capitalize() for name in CLASS_NAMES]
    probs = [result["probabilities"][name] for name in CLASS_NAMES]
    colors = ["#2ecc71" if name == result["prediction"] else "#bdc3c7" for name in CLASS_NAMES]

    bars = axes[1].barh(classes, probs, color=colors, edgecolor="black", linewidth=1.2)
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel("Probability", fontsize=11)
    axes[1].set_title(
        f"Prediction: {result['prediction'].capitalize()}\n"
        f"Confidence: {result['confidence']:.2%}",
        fontsize=12, fontweight="bold"
    )
    axes[1].grid(axis="x", alpha=0.3)

    # Add probability labels
    for bar, prob in zip(bars, probs):
        axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                     f"{prob:.2%}", va="center", fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "prediction_example.png", dpi=150, bbox_inches="tight")
    plt.show()


# --- Demo prediction on a test image ---
print("Testing prediction function...")
test_images = [path for path, _ in splits["test"]]
if test_images:
    demo_image = random.choice(test_images)
    result = predict(demo_image)
    print("\nPrediction Result:")
    print(json.dumps(result, indent=2))
    visualize_prediction(demo_image, result)



---

# Section 12: Export and Inference Script

### What we export:
1. **Model weights** — Saved to Google Drive as `classification_model.pth`
2. **Inference script** — Standalone Python script for backend integration
3. **Configuration** — JSON file with all parameters needed for inference

### Integration into Backend:
The saved model and inference code can be directly integrated into a Flask/FastAPI backend for the hackathon demo.


In [ ]:

# ============================================================
# 12.1 Save Model Weights
# ============================================================

# Save the final model to Google Drive
final_model_path = Path(config.CHECKPOINT_DIR) / "classification_model.pth"
torch.save(model.state_dict(), final_model_path)
print(f"Model weights saved to: {final_model_path}")

# Also save a complete checkpoint with metadata
final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
    "input_size": INPUT_SIZE,
    "normalize_mean": NORMALIZE_MEAN,
    "normalize_std": NORMALIZE_STD,
    "model_architecture": "EfficientNet-B0",
    "training_info": {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "best_val_acc": max(history_p1["val_acc"] + history_p2["val_acc"])
    }
}

checkpoint_path = Path(config.CHECKPOINT_DIR) / "final_checkpoint.pt"
torch.save(final_checkpoint, checkpoint_path)
print(f"Complete checkpoint saved to: {checkpoint_path}")

# Verify files
print("\nSaved files:")
for f in sorted(Path(config.CHECKPOINT_DIR).iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:40s} ({size_mb:.2f} MB)")


In [ ]:

# ============================================================
# 12.2 Save Inference Configuration
# ============================================================

import json

config_data = {
    "model": {
        "architecture": "EfficientNet-B0",
        "num_classes": NUM_CLASSES,
        "class_names": CLASS_NAMES,
        "class_to_idx": CLASS_TO_IDX,
        "input_size": INPUT_SIZE,
        "pretrained": True
    },
    "preprocessing": {
        "resize": INPUT_SIZE,
        "grayscale_to_rgb": True,
        "normalize_mean": NORMALIZE_MEAN,
        "normalize_std": NORMALIZE_STD
    },
    "classes": {
        name: {
            "index": idx,
            "description": {
                "normal": "Healthy breast tissue, no abnormalities",
                "benign": "Non-cancerous mass or abnormality",
                "malignant": "Cancerous tumor requiring treatment"
            }[name]
        }
        for name, idx in CLASS_TO_IDX.items()
    }
}

config_json_path = RESULTS_DIR / "inference_config.json"
with open(config_json_path, "w") as f:
    json.dump(config_data, f, indent=2)

print("Inference configuration saved:")
print(json.dumps(config_data, indent=2))


In [ ]:

# ============================================================
# 12.3 Generate Standalone Inference Script
# ============================================================
# This code writes a standalone inference script that can be
# used in the hackathon backend without the full notebook.

inference_script = r"""#!/usr/bin/env python3
\"\"\"
Standalone inference script for Breast Ultrasound Classification.
Integrate this into your hackathon backend API.

Usage:
    from inference import BreastUltrasoundPredictor
    predictor = BreastUltrasoundPredictor("path/to/checkpoint.pt")
    result = predictor.predict("path/to/image.png")
\"\"\"

import json
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
from typing import Dict


class BreastUltrasoundPredictor:
    \"\"\"
    Production-ready predictor for breast ultrasound classification.

    Args:
        checkpoint_path: Path to the model checkpoint (.pt file).
        device: torch device (auto-detected if None).
    \"\"\"

    CLASS_NAMES = ["normal", "benign", "malignant"]
    NUM_CLASSES = 3
    INPUT_SIZE = 224

    def __init__(self, checkpoint_path: str, device: str = None):
        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))
        self.checkpoint_path = Path(checkpoint_path)

        # Load checkpoint
        checkpoint = torch.load(self.checkpoint_path, map_location=self.device)

        # Extract config
        self.class_names = checkpoint.get("class_names", self.CLASS_NAMES)
        self.num_classes = checkpoint.get("num_classes", self.NUM_CLASSES)
        self.input_size = checkpoint.get("input_size", self.INPUT_SIZE)
        self.normalize_mean = checkpoint.get("normalize_mean", [0.5, 0.5, 0.5])
        self.normalize_std = checkpoint.get("normalize_std", [0.5, 0.5, 0.5])

        # Build model
        self.model = self._build_model()
        self.model.load_state_dict(checkpoint["model_state_dict"])
        self.model.to(self.device)
        self.model.eval()

        # Preprocessing pipeline
        self.transform = transforms.Compose([
            transforms.Resize((self.input_size, self.input_size)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.normalize_mean, std=self.normalize_std)
        ])

        print(f"Predictor loaded: {len(self.class_names)} classes")
        print(f"Device: {self.device}")

    def _build_model(self) -> nn.Module:
        \"\"\"Build EfficientNet-B0 with custom classifier head.\"\"\"
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, self.num_classes)
        )
        return model

    def predict(self, image_path: str) -> Dict:
        \"\"\"
        Predict the class of a breast ultrasound image.

        Args:
            image_path: Path to the image file.

        Returns:
            Dictionary with prediction, confidence, and probabilities.
        \"\"\"
        image = Image.open(image_path).convert("RGB")
        input_tensor = self.transform(image).unsqueeze(0).to(self.device)

        with torch.no_grad():
            outputs = self.model(input_tensor)
            probabilities = torch.softmax(outputs, dim=1)

        probs = probabilities[0].cpu().numpy()
        pred_idx = int(probs.argmax())

        return {
            "prediction": self.class_names[pred_idx],
            "confidence": round(float(probs[pred_idx]), 4),
            "probabilities": {
                name: round(float(prob), 4)
                for name, prob in zip(self.class_names, probs)
            }
        }

    def predict_batch(self, image_paths: list) -> list:
        \"\"\"Predict multiple images in a batch.\"\"\"
        return [self.predict(p) for p in image_paths]


# --- Example usage ---
if __name__ == "__main__":
    import sys

    if len(sys.argv) < 3:
        print("Usage: python inference.py <checkpoint.pt> <image.png>")
        sys.exit(1)

    ckpt_path = sys.argv[1]
    img_path = sys.argv[2]

    predictor = BreastUltrasoundPredictor(ckpt_path)
    result = predictor.predict(img_path)

    print(json.dumps(result, indent=2))
"""

# Write the script to file
script_path = RESULTS_DIR / "inference.py"
with open(script_path, "w") as f:
    f.write(inference_script)

print(f"Inference script saved to: {script_path}")
print("This script can be integrated into your hackathon backend.")



---

# Notebook Summary

## Completed Workflow

This notebook provided a complete end-to-end pipeline for breast ultrasound disease classification:

| Section | Description | Status |
|---------|------------|--------|
| 1 | Environment Setup | Complete |
| 2 | Google Drive Setup | Complete |
| 3 | Dataset Download & Inspection | Complete |
| 4 | Exploratory Data Analysis | Complete |
| 5 | Data Preprocessing | Complete |
| 6 | Model Selection | Complete |
| 7 | Build Classification Model | Complete |
| 8 | Training Pipeline | Complete |
| 9 | Model Evaluation | Complete |
| 10 | Error Analysis | Complete |
| 11 | Prediction Function | Complete |
| 12 | Export & Inference | Complete |

## Generated Artifacts

All outputs are saved in your Google Drive and local results folder:

| File | Location | Description |
|------|----------|-------------|
| `classification_model.pth` | Drive/BUSI_Checkpoints/ | Model weights |
| `final_checkpoint.pt` | Drive/BUSI_Checkpoints/ | Full checkpoint with config |
| `inference.py` | /content/results/ | Standalone inference script |
| `inference_config.json` | /content/results/ | Model configuration |
| `class_distribution.png` | /content/results/ | EDA visualization |
| `image_properties.png` | /content/results/ | EDA visualization |
| `pixel_statistics.png` | /content/results/ | EDA visualization |
| `image_quality.png` | /content/results/ | EDA visualization |
| `sample_images.png` | /content/results/ | EDA visualization |
| `mask_overlays.png` | /content/results/ | EDA visualization |
| `training_curves.png` | /content/results/ | Training visualization |
| `confusion_matrix.png` | /content/results/ | Evaluation visualization |

## Next Steps for Hackathon

1. **Integrate `inference.py`** into your FastAPI/Flask backend
2. **Test the prediction function** with new ultrasound images
3. **Consider improvements** — ensembles, attention, TTA
4. **Add explainability** — Grad-CAM heatmaps for clinical trust
5. **Deploy the model** with the inference script

## Model Architecture

- **Backbone:** EfficientNet-B0 (ImageNet pretrained)
- **Classifier:** Dropout(0.3) + Linear(1280, 3)
- **Input:** 224 x 224 x 3 (grayscale converted to RGB)
- **Classes:** Normal, Benign, Malignant
- **Parameters:** ~5.3M

## Training Strategy

- **Phase 1:** Frozen backbone, train classifier head (10 epochs)
- **Phase 2:** Unfrozen backbone, fine-tune entire network (20 epochs)
- **Optimizer:** AdamW with CosineAnnealingLR
- **Augmentation:** Flip, rotate, crop, brightness/contrast jitter
- **Class Balancing:** WeightedRandomSampler + class weights
